[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/alg/u4_espacios.ipynb)

# Álg. U4 · Espacios vectoriales y transformaciones lineales
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 4 del bloque de álgebra lineal. Tiene una sección por subtema, en el mismo orden que el libro:

1. Subespacios de $\mathbb R^n$ y combinaciones lineales
2. Independencia lineal
3. Base, dimensión, coordenadas y cambio de base
4. Transformaciones lineales: rotación, escalamiento y proyección

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección tiene una celda de teoría, código que la recorre paso a paso, una calculadora, sus casos de prueba y una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `ALG-U4` (bloque, unidad). Las celdas de código de cada sección vienen de `alg/u4_espacios/NN_*.py`; los fragmentos que el libro imprime, de `alg/u4_espacios/fragmentos/`.

**Convenciones.** En esta unidad hay dos maneras de escribir datos, y cada calculadora dice cuál usa.

- **Una lista de vectores** (secciones 1, 2 y 3) se escribe **un vector por renglón**, o con los vectores separados por `;`; las componentes se separan con comas o espacios. **Cada renglón escrito es un vector**, y la calculadora lo guarda como **una columna** de la matriz $A=[\,\mathbf v_1\ \mathbf v_2\ \cdots\ \mathbf v_k\,]$, de $n\times k$ ($n$ componentes, $k$ vectores). Por ejemplo, `1, 0, 1; 2, 1, 1` son los vectores $\mathbf v_1=(1,0,1)$ y $\mathbf v_2=(2,1,1)$, y

  $$A=\begin{bmatrix}1&2\\0&1\\1&1\end{bmatrix}.$$

  Se aceptan hasta 6 vectores de hasta 6 componentes, todos con el mismo número de componentes. También se acepta `(1, 0, 1), (2, 1, 1)` o `{(1, 0, 1), (2, 1, 1)}`. En la unidad 3 cada renglón era una ecuación; aquí cada renglón es un vector.
- **Una matriz** (secciones 3 y 4) se escribe **por renglones**, como en la unidad 3: cada renglón escrito es un renglón de la matriz. Por ejemplo, `1, 2; 3, 4` es $\begin{bmatrix}1&2\\3&4\end{bmatrix}$. **Ojo:** leído como lista de vectores, ese mismo texto da los vectores $(1,2)$ y $(3,4)$ como columnas, es decir, la transpuesta $\begin{bmatrix}1&3\\2&4\end{bmatrix}$. Hasta $6\times 6$.
- Un vector suelto ($\mathbf b$, $\mathbf x$, la dirección $\mathbf u$) se escribe como lista de números: `5, 4, 1`.
- Punto decimal. Puedes escribir fracciones como `2/3` (también `2 / 3`); los decimales se convierten a fracciones exactas (`0.1` se lee como $\tfrac{1}{10}$), así que las decisiones (rangos, independencia, coordenadas, núcleos) se toman sin error de redondeo. Se aceptan los signos menos tipográficos (− y –) y un espacio después del signo (`- 1`).
- Los vectores se nombran $\mathbf v_1, \mathbf v_2,\dots$ en el orden en que los escribes (en lo que imprimen las calculadoras, `v1`, `v2`, ...). Vectores, renglones y columnas se cuentan desde 1. Los parámetros de una solución general se llaman $s, t, r, p, q, k$, en el orden de las variables libres (no se usan $u$, $v$ ni $w$, para no confundirlos con los vectores).
- En lo que muestran las calculadoras, el signo menos es siempre `-`, las raíces se escriben con `√` y el seno con `sen`. Los decimales que se muestran se *redondean* (no se truncan) a las cifras significativas que elijas; si una fracción exacta resulta demasiado larga, la calculadora muestra solo sus decimales y lo avisa.
- Si usas las funciones desde código: a `lista_vectores` (y a las funciones que reciben vectores) le puedes pasar texto, una lista de vectores como `[(1, 0, 1), (2, 1, 1)]` o una `Matrix` de sympy; una `Matrix` se lee **por columnas** (cada columna es un vector), igual que la matriz $A$ que arma la calculadora.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re, io, textwrap, contextlib, time
from decimal import Decimal, ROUND_HALF_UP, localcontext
from fractions import Fraction
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registra la proyección "3d")
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import sympy as sp
import ipywidgets as widgets
from IPython.display import display, Math

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Utilidades compartidas por todo el notebook (adaptadas de la unidad 3): leer números y matrices exactos,
# redondear y mostrar
_NUMERO = re.compile(r"[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?")
_AYUDA_NUMERO = "usa enteros, fracciones como 2/3 o decimales con punto"


_LIMITE = Fraction(10) ** 15                  # entradas entre 1e-15 y 1e15 en valor absoluto (o 0)


def _en_rango(q, texto):
    if q != 0 and not Fraction(1) / _LIMITE <= abs(q) <= _LIMITE:
        raise ValueError(f"«{texto}» está fuera del intervalo que maneja esta calculadora: "
                         "usa valores entre 1e-15 y 1e15 en valor absoluto (o 0).")
    return sp.Rational(q.numerator, q.denominator)


def numero(valor, nombre="el número"):
    """Un número como racional exacto de sympy: '0.1' -> 1/10, '2/3' -> 2/3, 5 -> 5, 1.2 -> 6/5."""
    if valor is None:
        raise ValueError(f"{nombre} está vacío: escribe un número.")
    if isinstance(valor, bool):
        raise ValueError(f"«{valor}» no es un número: {_AYUDA_NUMERO}.")
    if isinstance(valor, (int, np.integer)):
        return _en_rango(Fraction(int(valor)), valor)
    if isinstance(valor, (float, np.floating)) or (isinstance(valor, sp.Basic) and valor.is_Float):
        v = float(valor)
        if not math.isfinite(v):
            raise ValueError(f"{nombre} no es finito (escribiste {v}).")
        return _en_rango(Fraction(repr(v)), repr(v))      # repr da el decimal más corto: 0.1 -> '0.1' -> 1/10 exacto
    if isinstance(valor, sp.Basic):
        if valor.is_Rational:
            return valor                             # ya es exacto (por ejemplo, un resultado intermedio): no se acota
        raise ValueError(f"«{valor}» no es un número racional: {_AYUDA_NUMERO}.")
    texto = re.sub(r"\s+", "", str(valor)).replace("−", "-").replace("–", "-")      # acepta los signos menos tipográficos
    partes = texto.split("/")
    formas = [_NUMERO.fullmatch(p) for p in partes]
    if not texto or len(partes) > 2 or not all(formas):
        raise ValueError(f"«{str(valor).strip()}» no es un número: {_AYUDA_NUMERO}." if texto else
                         f"{nombre} está vacío: escribe un número.")
    if any(f.group(2) and abs(int(f.group(2)[1:])) > 30 for f in formas):   # 1e999999 no se calcula: se rechaza antes
        raise ValueError(f"«{texto}» está fuera del intervalo que maneja esta calculadora: "
                         "usa valores entre 1e-15 y 1e15 en valor absoluto (o 0).")
    num, *den = (Fraction(p) for p in partes)
    if den and den[0] == 0:
        raise ValueError(f"«{texto}» divide entre cero.")
    return _en_rango(num / den[0] if den else num, texto)


def _entradas(k):
    return f"{k} entrada" if k == 1 else f"{k} entradas"


def _componentes(k):
    return f"{k} componente" if k == 1 else f"{k} componentes"


def _desde_renglones(renglones, nombre):
    """Lista de renglones (cada uno, lista de entradas) -> Matrix de racionales, con mensajes claros."""
    if not renglones or not renglones[0]:
        raise ValueError(f"escribe {nombre} por renglones, por ejemplo: 1, 2; 3, 4")
    n = len(renglones[0])
    salida = []
    for i, renglon in enumerate(renglones, 1):
        if len(renglon) != n:
            raise ValueError(f"en {nombre}, el renglón {i} tiene {_entradas(len(renglon))} y el 1 tiene {n}: "
                             "todos los renglones deben tener el mismo número de entradas")
        fila = []
        for j, v in enumerate(renglon, 1):
            try:
                fila.append(numero(v))
            except ValueError as err:
                raise ValueError(f"en {nombre}, renglón {i}, columna {j}: {err}") from None
        salida.append(fila)
    return sp.Matrix(salida)


def _normaliza_entrada(texto):
    """Signos menos tipográficos (− y –) como '-', llaves como paréntesis y sin espacios alrededor de '/' ni entre un
    signo y su número: '2 / 3' -> '2/3', '- 1' -> '-1', '{(1, 1), (0, 1)}' -> '((1, 1), (0, 1))'."""
    t = texto.replace("−", "-").replace("–", "-").replace("{", "(").replace("}", ")")
    t = re.sub(r"\s*/\s*", "/", t)
    return re.sub(r"(^|[\s,;(\[])([+-])\s+(?=[\d.])", r"\1\2", t)


def _trozos(texto):
    """Texto -> lista de renglones, cada uno lista de entradas (texto). Acepta ';' o saltos de línea entre renglones,
    comas o espacios entre entradas, y la forma de Python [[1, 2], [3, 4]], (1, 2), (3, 4) o {(1, 2), (3, 4)}.
    Devuelve también el número del primer renglón con una entrada vacía (o None)."""
    t = _normaliza_entrada(texto)
    t = re.sub(r"[\]\)]\s*,?\s*[\[\(]", ";", t)            # [[1, 2], [3, 4]] o (1, 2), (3, 4): cada grupo es un renglón
    t = re.sub(r"[\[\]\(\)]", " ", t)
    renglones = [r.strip() for r in re.split(r"[;\n]", t)]
    renglones = [r for r in renglones if r]
    entradas, vacia = [], None
    for i, r in enumerate(renglones, 1):
        partes = re.split(r"\s*,\s*|\s+", r)
        if "" in partes and vacia is None:
            vacia = i
        entradas.append(partes)
    return entradas, vacia


def matriz(texto, nombre="la matriz"):
    """Lee una matriz escrita por renglones ('1, 2; 3, 4' o un renglón por línea) como Matrix de racionales exactos."""
    if not isinstance(texto, str):
        return como_matriz(texto, nombre)
    entradas, vacia = _trozos(texto)
    if vacia is not None:
        raise ValueError(f"en {nombre}, el renglón {vacia} tiene una entrada vacía: "
                         "revisa que no haya dos comas seguidas ni una coma al principio o al final")
    try:
        return _desde_renglones(entradas, nombre)
    except ValueError as err:
        if "todos los renglones deben tener" in str(err):
            raise ValueError(f"{err}{pista_coma(texto)}") from None
        raise


def pista_coma(texto):
    """La pista de usar punto decimal, solo si la lectura es de veras ambigua: hay comas entre dígitos sin espacio
    (1,5) y además números separados solo por espacios (1,5 2,5), que es como se escriben decimales con coma."""
    if isinstance(texto, str) and re.search(r"\d,\d", texto) and re.search(r"\d[ \t]+[-+.]?\d", texto):
        return "; si escribiste decimales, usa punto decimal: 1.5, no 1,5"
    return ""


def como_matriz(A, nombre="la matriz"):
    """Acepta texto, Matrix de sympy, lista de listas o arreglo de numpy y devuelve una Matrix de racionales exactos."""
    if isinstance(A, str):
        return matriz(A, nombre)
    if isinstance(A, sp.MatrixBase):
        renglones = A.tolist()
    elif isinstance(A, np.ndarray):
        if A.ndim > 2:
            raise ValueError(f"{nombre} debe ser una tabla de números (renglones y columnas)")
        renglones = np.atleast_2d(A).tolist()
    elif isinstance(A, (list, tuple)):
        anidada = [isinstance(f, (list, tuple, np.ndarray, sp.MatrixBase)) for f in A]
        if A and all(anidada):
            renglones = [list(f) for f in A]
        elif not any(anidada):
            renglones = [list(A)]                  # una lista sola es un renglón
        else:
            raise ValueError(f"en {nombre}, mezclaste números sueltos con renglones")
    else:
        renglones = [[A]]
    return _desde_renglones(renglones, nombre)


def vector(texto, nombre="el vector"):
    """Lee un vector ('1, 2, 3', '1; 2; 3', (1, 2, 3) o [1, 2, 3]) como columna de racionales exactos.
    Los mensajes de error dicen qué componente está mal."""
    ejemplo = f"escribe {nombre} como lista de números, por ejemplo: 5, 4, 1"
    if texto is None:
        raise ValueError(ejemplo)
    if isinstance(texto, str):
        filas, vacia = _trozos(texto)
        if vacia is not None:
            raise ValueError(f"en {nombre} hay una componente vacía: revisa que no haya dos comas seguidas "
                             "ni una coma al principio o al final")
        if not filas:
            raise ValueError(ejemplo)
        if len(filas) == 1:
            entradas = filas[0]
        elif all(len(f) == 1 for f in filas):
            entradas = [f[0] for f in filas]
        else:
            raise ValueError(f"{nombre} debe ser una lista de números, por ejemplo: 5, 4, 1 (escribiste varios renglones)")
    elif isinstance(texto, (list, tuple, np.ndarray, sp.MatrixBase)):
        plano = ((isinstance(texto, np.ndarray) and texto.ndim == 1) or
                 (isinstance(texto, (list, tuple)) and
                  not any(isinstance(e, (list, tuple, np.ndarray, sp.MatrixBase)) for e in texto)))
        if not plano:                                   # Matrix, arreglo 2D o lista de listas
            return _vector_de_matriz(texto, nombre, ejemplo)
        entradas = list(texto)
        if not entradas:
            raise ValueError(ejemplo)
    else:
        entradas = [texto]                              # un número solo
    valores = []
    for j, e in enumerate(entradas, 1):
        try:
            valores.append(numero(e))
        except ValueError as err:
            raise ValueError(f"en {nombre}, componente {j}: {err}") from None
    return sp.Matrix(valores)


def _vector_de_matriz(texto, nombre, ejemplo):
    """Un vector dado como Matrix, arreglo o lista anidada: debe tener un solo renglón o una sola columna."""
    try:
        v = como_matriz(texto, nombre)
    except ValueError as err:
        if str(err).startswith("escribe "):
            raise ValueError(ejemplo) from None
        raise
    if v.rows == 1 or v.cols == 1:
        return v.reshape(v.rows * v.cols, 1)
    raise ValueError(f"{nombre} debe ser una lista de números, por ejemplo: 5, 4, 1 (escribiste una matriz {dims(v)})")


def dims(M):
    """Tamaño 'm×n' de una matriz."""
    return f"{M.rows}×{M.cols}"


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales.
    Un racional exacto se redondea desde su valor exacto (p/q con 50 cifras), no desde su aproximación en punto
    flotante: 1249999999999999999/10^19 con 2 cifras da 0.12, no 0.13. Una expresión exacta con raíces, senos o
    cosenos se evalúa con 50 cifras antes de redondear."""
    with localcontext() as ctx:
        ctx.prec = 50
        if isinstance(valor, sp.Basic) and not valor.is_Rational and valor.is_number:
            valor = sp.N(valor, 50)
            if valor.is_real is False:
                return "no definido"
            d = Decimal(str(valor)) if valor.is_Float else Decimal(repr(float(valor)))
        elif isinstance(valor, (sp.Rational, Fraction, int, np.integer)) and not isinstance(valor, bool):
            p, q = (int(valor.p), int(valor.q)) if isinstance(valor, sp.Rational) else (
                (valor.numerator, valor.denominator) if isinstance(valor, Fraction) else (int(valor), 1))
            d = Decimal(p) / Decimal(q)
        else:
            v = float(valor)
            if math.isnan(v):
                return "no definido"
            if math.isinf(v):
                return "+∞" if v > 0 else "-∞"
            d = Decimal(repr(v))
        if d == 0:
            return "0"
        q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
        if q.adjusted() != d.adjusted():      # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
            q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
        e = q.adjusted()
        if e > 15 or e < -4:                  # muy grandes o muy chicos: notación científica, 1.234×10^-7
            return f"{format(q.scaleb(-e), 'f')}×10^{e}"
        if q.adjusted() >= n:                 # enteros grandes: sin notación científica
            return f"{int(q):d}"
        return format(q, "f")


def decimales_fijos(valor, d=2):
    """Redondea (5 sube) a d decimales fijos, desde el valor exacto: sqrt(3) con 2 -> '1.73'; 1 -> '1.00'."""
    with localcontext() as ctx:
        ctx.prec = 50
        x = Decimal(str(sp.N(sp.sympify(valor), 40)))
        return format(x.quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP), "f")


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una función rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def _mensaje(fn):
    """Texto del ValueError que lanza fn() ('' si no lanza): para probar que el mensaje explica el error."""
    try:
        fn()
    except ValueError as err:
        return str(err)
    return ""


# --- formato de números y matrices ---------------------------------------------------------------
_SUB = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
_SUP = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")


def _sub(*indices):
    """Subíndices: _sub(1, 2) -> '₁₂'; con índices de dos cifras los separa con coma."""
    if all(k < 10 for k in indices):
        return "".join(str(k) for k in indices).translate(_SUB)
    return "₍" + ",".join(str(k) for k in indices).translate(_SUB) + "₎"


def _R(n):
    """Nombre del espacio en texto plano: _R(3) -> 'R³'."""
    return "R" + str(n).translate(_SUP)


def _txt(e):
    """Una cantidad exacta como texto legible: sqrt(3)/2 -> √3/2, 2*sin(30) -> 2·sen(30), pi/6 -> π/6."""
    t = str(e)
    t = re.sub(r"sqrt\((\d+)\)", r"√\1", t).replace("sqrt(", "√(")
    return t.replace("**", "^").replace("*", "·").replace("sin(", "sen(").replace("pi", "π")


def _latex(x):
    """LaTeX de sympy con la convención del libro: sen en lugar de sin."""
    return sp.latex(x).replace(r"\sin", r"\operatorname{sen}")


def _exacta_legible(M, largo=28, total=400):
    """True si la forma exacta de M se lee bien: cada entrada con pocos caracteres y no demasiadas en total.
    Si no, las calculadoras muestran solo los decimales y lo avisan."""
    textos = [_txt(e) for e in M]
    return all(len(t) <= largo for t in textos) and sum(len(t) for t in textos) <= total


_LARGA = "la forma exacta es muy larga: se muestra redondeada"


def _igual(nombre, x, n=4):
    """'nombre = x' con x exacto (y sus decimales si no es entero); si la forma exacta es muy larga,
    'nombre ≈ decimales (aviso)'."""
    x = sp.sympify(x)
    if len(_txt(x)) > 28:
        return f"{nombre} ≈ {cifras(x, n)} ({_LARGA})"
    return f"{nombre} = {_txt(x)}" + ("" if x.is_integer else f" ≈ {cifras(x, n)}")


def _en_linea(M):
    """Matriz en una línea con la misma notación de entrada: [2, 3; 5, -2]."""
    return "[" + "; ".join(", ".join(_txt(e) for e in M.row(i)) for i in range(M.rows)) + "]"


def _tupla(v, n=None):
    """Vector como (a, b, c); con n, además sus decimales redondeados a n cifras. Si la forma exacta es muy larga,
    solo los decimales (a 4 cifras si no se da n), con el aviso."""
    if not _exacta_legible(v):
        return "≈ (" + ", ".join(cifras(e, n or 4) for e in v) + f") ({_LARGA})"
    exacto = "(" + ", ".join(_txt(e) for e in v) + ")"
    if n is None or all(e.is_integer for e in v):
        return exacto
    return exacto + " ≈ (" + ", ".join(cifras(e, n) for e in v) + ")"


def _aprox(v, n):
    """Decimales redondeados de un vector: '(0.5000, 1.333)'; con una sola entrada, sin paréntesis."""
    textos = [cifras(e, n) for e in v]
    return textos[0] if len(textos) == 1 else "(" + ", ".join(textos) + ")"


def _plano(M, fmt=None, corte=None):
    """Texto plano de una matriz con columnas alineadas; corte = columna tras la que se dibuja una barra."""
    fmt = fmt or _txt
    celdas = [[fmt(M[i, j]) for j in range(M.cols)] for i in range(M.rows)]
    anchos = [max(len(celdas[i][j]) for i in range(M.rows)) for j in range(M.cols)]
    lineas = []
    for fila in celdas:
        partes = [c.rjust(a) for c, a in zip(fila, anchos)]
        if corte:
            partes = partes[:corte] + ["|"] + partes[corte:]
        lineas.append("[ " + "  ".join(partes) + " ]")
    return "\n".join(lineas)


_TEX = [("⁻¹", "^{-1}"), ("ᵀ", "^{T}"), ("²", "^{2}"), ("·", r"\cdot "), ("×", r"\times "), ("≈", r"\approx "),
        ("←", r"\leftarrow "), ("−", "-"), ("|", r"\,|\,")]
_SUB_TEX = re.compile("[₀₁₂₃₄₅₆₇₈₉]+")
_PROSA = re.compile(r"[A-Za-zÁÉÍÓÚáéíóúñü]{3,}")


def _a_tex(nombre):
    """Nombre de una cantidad en LaTeX. Si lleva palabras ('forma escalonada reducida'), va completo en \\text{...},
    con sus espacios. Si no, se traducen los símbolos (P_C⁻¹·P_B -> P_C^{-1}\\cdot P_B) y los subíndices (A2 -> A_{2},
    v1 -> \\mathbf v_{1}). Las calculadoras pasan el LaTeX a mano cuando el nombre mezcla palabras y símbolos."""
    if _PROSA.search(re.sub(r"\bdet\b", "", nombre)):
        return r"\text{" + nombre.replace("{", r"\{").replace("}", r"\}") + "}"
    t = nombre
    for a, b in _TEX:
        t = t.replace(a, b)
    t = re.sub(r"\bdet\s+", r"\\det ", t)
    t = _SUB_TEX.sub(lambda m: "_{" + m.group(0).translate(str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")) + "}", t)
    return re.sub(r"(?<![A-Za-z0-9\\])([A-Za-z])(\d+)(?![\d/.])",
                  lambda m: (r"\mathbf " if m.group(1) in "vbeu" else "") + m.group(1) + "_{" + m.group(2) + "}", t)


def _tex_decimales(M, n):
    filas = r" \\ ".join(" & ".join(cifras(M[i, j], n) for j in range(M.cols)) for i in range(M.rows))
    return r"\left[\begin{matrix}" + filas + r"\end{matrix}\right]"


class _Formula(Math):
    """Math de IPython con una versión en texto plano (la que se ve donde no se dibuja LaTeX)."""
    def __init__(self, tex, texto):
        super().__init__(tex)
        self._texto = texto

    def __repr__(self):
        return self._texto


class _Bloque:
    """Salida en Markdown (texto en recuadro y fórmulas en LaTeX) con su versión en texto plano."""
    def __init__(self, partes):
        self._partes = partes

    @staticmethod
    def _envuelve(texto, ancho=100):
        """Corta los renglones largos (en un recuadro de código no se ajustan solos); conserva la sangría."""
        lineas = []
        for linea in texto.rstrip("\n").split("\n"):
            sangria = " " * (len(linea) - len(linea.lstrip(" ")))
            lineas += textwrap.wrap(linea, ancho, subsequent_indent=sangria + "  ") if len(linea) > ancho else [linea]
        return "\n".join(lineas)

    def _repr_markdown_(self):
        return "\n\n".join(f"$${tex}$$" if tipo == "tex" else "```text\n" + self._envuelve(texto) + "\n```"
                            for tipo, tex, texto in self._partes)

    def __repr__(self):
        return "\n".join(texto.rstrip("\n") for _, _, texto in self._partes)


_BLOQUES = []      # bloques abiertos: lo que se muestra dentro de uno se junta y se entrega de una vez


@contextlib.contextmanager
def bloque():
    """Junta lo que se imprime y las fórmulas que se muestran dentro del bloque y lo entrega como UNA sola salida.
    Las calculadoras lo usan para que su resultado aparezca completo de una vez, en el orden en que se produjo."""
    partes, buf = [], io.StringIO()

    def vacia():
        if buf.getvalue():
            partes.append(("texto", "", buf.getvalue().replace("= ≈", "≈")))     # un valor que salió redondeado
            buf.seek(0); buf.truncate()

    _BLOQUES.append((partes, vacia))
    try:
        with contextlib.redirect_stdout(buf):
            yield
    finally:
        vacia()
        _BLOQUES.pop()
        if partes:
            _emite(_Bloque(partes)) if _BLOQUES else display(_Bloque(partes))


def _emite(formula):
    """Muestra una fórmula, o la agrega al bloque abierto si lo hay."""
    if not _BLOQUES:
        display(formula)
        return
    partes, vacia = _BLOQUES[-1]
    vacia()
    if isinstance(formula, _Bloque):
        partes.extend(formula._partes)
    else:
        partes.append(("tex", formula.data, formula._texto))


def muestra(nombre, M, n=None, tex=None):
    """Muestra 'nombre = M' con LaTeX (con su tamaño si es matriz). Con n agrega los decimales redondeados a n cifras.
    tex es el nombre en LaTeX (si no se da, se arma con _a_tex). Si la forma exacta es muy larga, solo los decimales."""
    tex_nombre = tex if tex is not None else _a_tex(nombre)
    if isinstance(M, sp.MatrixBase):
        if _exacta_legible(M):
            tex = f"{tex_nombre} = {_latex(M)}"
            texto = f"{nombre}  ({dims(M)}) =\n{_plano(M)}"
            if n is not None and not all(e.is_integer for e in M):
                tex += r" \approx " + _tex_decimales(M, n)
                texto += f"\n≈\n{_plano(M, lambda e: cifras(e, n))}"
        else:
            tex = f"{tex_nombre} \\approx {_tex_decimales(M, n or 4)}" + r"\quad\text{(" + _LARGA + ")}"
            texto = f"{nombre}  ({dims(M)}) ≈\n{_plano(M, lambda e: cifras(e, n or 4))}\n({_LARGA})"
        tex += rf"\qquad ({M.rows}\times {M.cols})"
    else:
        M = sp.sympify(M)
        tex, texto = f"{tex_nombre} = {_latex(M)}", f"{nombre} = {_txt(M)}"
        if n is not None and M.is_number and not M.is_integer:
            tex += rf" \approx {cifras(M, n)}"
            texto += f" ≈ {cifras(M, n)}"
    _emite(_Formula(tex, texto))

In [ ]:
# Utilidades de esta unidad: listas de vectores (cada vector es una COLUMNA), eliminación exacta, núcleos,
# relaciones de dependencia y textos de combinaciones lineales
_MAX_VECTORES, _MAX_COMPONENTES = 6, 6        # hasta 6 vectores de hasta 6 componentes; matrices de hasta 6×6


def lista_vectores(texto, nombre="la lista de vectores"):
    """Lee una lista de vectores escrita UN VECTOR POR RENGLÓN o con los vectores separados por ';'
    ('1, 0, 1; 2, 1, 1') y devuelve la matriz A de n×k que tiene esos vectores como COLUMNAS.
    También acepta una lista de vectores ([(1, 0, 1), (2, 1, 1)] o ['1, 0, 1', '2, 1, 1']) y una Matrix de sympy,
    que se toma tal cual: sus columnas ya son los vectores."""
    if isinstance(texto, sp.MatrixBase):
        A = como_matriz(texto, nombre)
        lista = [list(A[:, j]) for j in range(A.cols)]
    elif isinstance(texto, str):
        lista, vacia = _trozos(texto)
        if vacia is not None:
            raise ValueError(f"en {nombre}, el vector {vacia} tiene una componente vacía: "
                             "revisa que no haya dos comas seguidas ni una coma al principio o al final")
    elif isinstance(texto, (list, tuple)):
        lista = []
        for i, v in enumerate(texto, 1):
            if isinstance(v, (str, list, tuple, np.ndarray, sp.MatrixBase)):
                lista.append(list(vector(v, f"el vector {i} de {nombre}")))
            else:
                raise ValueError(f"en {nombre}, escribe cada vector como una lista de números, por ejemplo: "
                                 "[(1, 0, 1), (2, 1, 1)]")
    else:
        raise ValueError(f"escribe {nombre} con un vector por renglón o separados por «;», por ejemplo: 1, 0, 1; 2, 1, 1")
    if not lista or not lista[0]:
        raise ValueError(f"escribe {nombre} con un vector por renglón o separados por «;», por ejemplo: 1, 0, 1; 2, 1, 1")
    k, n = len(lista), len(lista[0])
    for i, v in enumerate(lista, 1):
        if len(v) != n:
            raise ValueError(f"en {nombre}, el vector {i} tiene {_componentes(len(v))} y el vector 1 tiene {n}: "
                             f"todos los vectores deben tener el mismo número de componentes{pista_coma(texto)}")
    if k > _MAX_VECTORES:
        raise ValueError(f"{nombre} tiene {k} vectores; esta calculadora acepta hasta {_MAX_VECTORES} "
                         "(recuerda: cada renglón escrito es un vector)")
    if n > _MAX_COMPONENTES:
        raise ValueError(f"en {nombre}, los vectores tienen {n} componentes; esta calculadora acepta hasta "
                         f"{_MAX_COMPONENTES}{pista_coma(texto)}")
    columnas = []
    for i, v in enumerate(lista, 1):
        col = []
        for j, e in enumerate(v, 1):
            try:
                col.append(numero(e))
            except ValueError as err:
                raise ValueError(f"en {nombre}, vector {i}, componente {j}: {err}") from None
        columnas.append(col)
    return sp.Matrix(columnas).T


def _revisa_tamano(M, nombre):
    """Rechaza matrices de más de 6 renglones o 6 columnas."""
    if M.rows > _MAX_COMPONENTES or M.cols > _MAX_VECTORES:
        raise ValueError(f"{nombre} es de {dims(M)}; esta calculadora acepta matrices de hasta 6×6")


def matriz_limitada(texto, nombre="A"):
    """Una matriz escrita POR RENGLONES, de hasta 6×6, con los mismos mensajes que matriz()."""
    M = como_matriz(texto, nombre)
    try:
        _revisa_tamano(M, nombre)
    except ValueError as err:
        raise ValueError(f"{err}{pista_coma(texto)}") from None
    return M


def nombres_incognitas(n):
    """x, y, z con tres incógnitas o menos; x₁, ..., xₙ con más."""
    return ["x", "y", "z"][:n] if n <= 3 else [f"x{_sub(k)}" for k in range(1, n + 1)]


def parametros(k):
    """Nombres de k parámetros, en el orden de las variables libres: s, t, r, p, q, k (con más de seis: s₁, s₂, ...).
    No se usan u, v ni w, para no confundirlos con los vectores."""
    if k == 0:
        return []
    if k <= 6:
        return list(sp.symbols("s t r p q k")[:k])
    return list(sp.symbols(" ".join(f"s{_sub(i)}" for i in range(1, k + 1))))


def _es_cero(e):
    """True si e vale 0. Un racional se compara directo; una expresión exacta (con raíces, senos o cosenos) se
    simplifica y, si sympy no la reduce, se evalúa con 50 cifras."""
    e = sp.sympify(e)
    if e.is_Rational:
        return e == 0
    v = sp.simplify(e)
    if v == 0:
        return True
    if v.is_number:
        return bool(abs(sp.N(v, 50)) < sp.Float("1e-40", 50))
    return False


def _simplifica(M):
    """Simplifica las entradas que no son racionales (las racionales se quedan como están)."""
    return M.applyfunc(lambda e: e if e.is_Rational else sp.simplify(e))


def _factor(a):
    """Coeficiente delante de un renglón: 1 -> '', 2 -> '2·', 1/2 -> '(1/2)·', -1 -> '(-1)·'."""
    if a == 1:
        return ""
    if a.is_integer and a > 0:
        return f"{a}·"
    return f"({a})·"


def _op_suma(i, m, k):
    """Texto de R_i ← R_i - m·R_k con el signo que toca: m = 2 -> 'R2 ← R2 - 2·R1'; m = -3 -> 'R2 ← R2 + 3·R1'."""
    if m.is_number and m.is_real and m < 0:
        return f"R{i} ← R{i} + {_factor(-m)}R{k}"
    return f"R{i} ← R{i} - {_factor(m)}R{k}"


def _elimina(M, n_a, con_b=True):
    """Eliminación de Gauss y después de Jordan, exacta, sobre una copia de M (adaptada de la unidad 3).

    Los pivotes se buscan en las primeras n_a columnas; se intercambian renglones solo si el pivote es 0 y abajo hay
    una entrada distinta de cero (como a mano). Con con_b, al final se busca un pivote en la columna n_a (la de b):
    si aparece, hay un renglón 0 = c. Funciona también con entradas exactas no racionales (senos y cosenos de una
    rotación): para decidir si una entrada es cero, primero se simplifica.
    Devuelve un diccionario: E (escalonada), R (reducida), pivotes (columnas con pivote, desde 0), libres, r (rango),
    r_aum (rango con la columna b), imposible ((renglón, c) o None) y pasos, una lista de (tipo, texto, matriz)."""
    E = M.copy()
    racional = all(e.is_Rational for e in E)
    limpia = (lambda X: X) if racional else _simplifica
    m = E.rows
    pasos = [("fase", "Gauss: ceros debajo de cada pivote (forma escalonada)", None)]
    pivotes, libres, f, imposible = [], [], 0, None
    for c in range(n_a + (1 if con_b and M.cols > n_a else 0)):
        es_b = c == n_a
        k = next((i for i in range(f, m) if not _es_cero(E[i, c])), None)
        if k is None:
            if not es_b:
                libres.append(c)
                motivo = "ya no quedan renglones abajo" if f >= m else f"de R{f + 1} hacia abajo todo es cero"
                pasos.append(("nota", f"la columna {c + 1} no tiene pivote: {motivo}", None))
            continue
        if k != f:
            pasos.append(("nota", f"el pivote de la columna {c + 1} es cero; se intercambian R{f + 1} y R{k + 1}", None))
            E.row_swap(f, k)
            pasos.append(("op", f"R{f + 1} ↔ R{k + 1}", E.copy()))
        for i in range(f + 1, m):
            if not _es_cero(E[i, c]):
                mult = limpia(sp.Matrix([E[i, c] / E[f, c]]))[0]
                E[i, :] = limpia(E[i, :] - mult * E[f, :])
                E[i, c] = 0
                pasos.append(("op", _op_suma(i + 1, mult, f + 1), E.copy()))
        if es_b:
            imposible = (f + 1, E[f, c])
            pasos.append(("nota", f"R{f + 1} quedó como 0 = {E[f, c]}: ninguna combinación lo cumple", None))
        else:
            pivotes.append(c)
        f += 1
    if len(pasos) == 1:
        pasos.append(("nota", "la matriz ya estaba escalonada: no hace falta ninguna operación", None))
    R = E.copy()
    pasos.append(("fase", "Jordan: cada pivote en 1 y ceros también arriba (forma escalonada reducida)", None))
    hechas = len(pasos)
    todos = list(enumerate(pivotes)) + ([(len(pivotes), n_a)] if imposible is not None else [])
    for f, c in reversed(todos):
        if not _es_cero(R[f, c] - 1):
            k = limpia(sp.Matrix([1 / R[f, c]]))[0]
            R[f, :] = limpia(k * R[f, :])
            R[f, c] = 1
            pasos.append(("op", f"R{f + 1} ← {_factor(k)}R{f + 1}", R.copy()))
        for i in range(f):
            if not _es_cero(R[i, c]):
                mult = R[i, c]
                R[i, :] = limpia(R[i, :] - mult * R[f, :])
                R[i, c] = 0
                pasos.append(("op", _op_suma(i + 1, mult, f + 1), R.copy()))
    if len(pasos) == hechas:
        pasos.append(("nota", "la forma escalonada ya era reducida: no hace falta ninguna operación", None))
    return {"E": E, "R": R, "pivotes": pivotes, "libres": libres, "r": len(pivotes),
            "r_aum": len(pivotes) + (imposible is not None), "imposible": imposible, "pasos": pasos}


def _imprime_pasos(pasos, corte=None, sangria="  ", n_cifras=4):
    for tipo, texto, M in pasos:
        if tipo == "fase":
            print(f"— {texto} —")
        elif tipo == "nota":
            print(f"{sangria}Nota: {texto}.")
        else:
            legible = _exacta_legible(M)
            print(f"{sangria}{texto}" + ("" if legible else f"   ({_LARGA} a {n_cifras} cifras)"))
            plano = _plano(M, None if legible else (lambda e: cifras(e, n_cifras)), corte=corte)
            print("\n".join(sangria + "  " + linea for linea in plano.split("\n")))


def _solo_gauss(pasos):
    """Los pasos de la fase de Gauss (hasta la forma escalonada), sin los de Jordan."""
    fases = [k for k, (tipo, _, _) in enumerate(pasos) if tipo == "fase"]
    return pasos[:fases[1]] if len(fases) > 1 else pasos


def solucion_general(R, pivotes, n):
    """Solución de un sistema compatible a partir de su forma reducida R (con la columna b al final).
    Devuelve (x, particular, direcciones, parámetros): cada variable libre es un parámetro (s, t, u, ... en el orden
    de las libres), x = particular + s·d₁ + t·d₂ + ..., y la particular es la que tiene las libres en 0."""
    libres = [c for c in range(n) if c not in pivotes]
    ps = parametros(len(libres))
    particular = sp.zeros(n, 1)
    for f, c in enumerate(pivotes):
        particular[c] = R[f, n]
    direcciones = _direcciones(R, pivotes, n)
    x = particular + sum((p * d for p, d in zip(ps, direcciones)), sp.zeros(n, 1))
    return x, particular, direcciones, ps


def _direcciones(R, pivotes, n):
    """Base de Nul(A) leída de la forma reducida R de A (sus primeras n columnas): un vector por variable libre,
    con esa libre en 1 y las demás libres en 0."""
    direcciones = []
    for c_libre in [c for c in range(n) if c not in pivotes]:
        d = sp.zeros(n, 1)
        d[c_libre] = 1
        for f, c in enumerate(pivotes):
            d[c] = -R[f, c_libre]
        direcciones.append(d)
    return direcciones


def base_nulo(M):
    """Base exacta de Nul(M) = {x : M x = 0} (lista vacía si solo contiene al 0)."""
    el = _elimina(M, M.cols, con_b=False)
    return _direcciones(el["R"], el["pivotes"], M.cols)


def _primitivo(v):
    """El vector racional v (no nulo) escalado a enteros sin factor común y con su primera entrada distinta de cero
    positiva: (1/2, -1, 0) -> (1, -2, 0); (-2, -1, 1) -> (2, 1, -1)."""
    v = sp.Matrix(v)
    m = math.lcm(*[int(sp.Rational(e).q) for e in v])
    w = [int(e * m) for e in v]
    g = math.gcd(*w)
    signo = 1 if next(e for e in w if e != 0) > 0 else -1
    return sp.Matrix([signo * e // g for e in w])


def _coeficiente(a, letra, primero):
    """Un término 'a·letra' con el signo que toca: (-2, 's', False) -> '- 2s'; (1/4, 's', True) -> '(1/4)s'."""
    mag = abs(a)
    coef = "" if (mag == 1 and letra) else (f"{mag}" if mag.is_integer else f"({mag})")
    if primero:
        return f"{'-' if a < 0 else ''}{coef}{letra}"
    return f"{'-' if a < 0 else '+'} {coef}{letra}"


def _lineal(e, ps):
    """Expresión lineal en los parámetros, con la constante primero: '4 - 2s', '(1/4)s', '7 - 2s - 3t'."""
    e = sp.expand(sp.sympify(e))
    constante = e.subs({p: 0 for p in ps})
    partes = [str(constante)] if constante != 0 else []
    for p in ps:
        a = e.coeff(p)
        if a != 0:
            partes.append(_coeficiente(a, str(p), not partes))
    return " ".join(partes) if partes else "0"


def _termino(c, nombre, primero):
    """'2·v1', '- 1·v3', '+ (1/2)·v2': el coeficiente siempre se escribe, también el 1 y el 0. Si la fracción exacta
    es muy larga, se escribe redondeada: '+ (≈1.234×10^5)·v2'."""
    c = sp.sympify(c)
    mag = abs(c)
    coef = _txt(mag) if mag.is_integer else f"({_txt(mag)})"
    if len(coef) > 30:
        coef = f"(≈{cifras(mag, 4)})"
    negativo = bool(c.is_real and c < 0)
    if primero:
        return f"{'-' if negativo else ''}{coef}·{nombre}"
    return f"{'-' if negativo else '+'} {coef}·{nombre}"


def texto_relacion(c, nombres=None):
    """Relación de dependencia c₁v₁ + ... + c_kv_k = 0 como texto, sin los términos con coeficiente 0:
    (2, 1, -1) -> '2·v1 + 1·v2 - 1·v3 = 0'."""
    nombres = nombres or [f"v{j}" for j in range(1, len(c) + 1)]
    partes = []
    for cj, nombre in zip(c, nombres):
        if cj != 0:
            partes.append(_termino(cj, nombre, not partes))
    return (" ".join(partes) if partes else "0") + " = 0"


def _texto_combinacion(coef, ps=(), nombres=None):
    """Combinación c₁·v1 + c₂·v2 + ... con todos sus términos: '-3·v1 + 4·v2'; con parámetros,
    '(2 - s)·v1 + (3 - s)·v2 + (4 - s)·v3 + s·v4' (la constante primero)."""
    nombres = nombres or [f"v{j}" for j in range(1, len(coef) + 1)]
    partes = []
    for c, nombre in zip(coef, nombres):
        c = sp.sympify(c)
        if c.free_symbols:
            texto = f"{c}·{nombre}" if c.is_Symbol else f"({_lineal(c, ps)})·{nombre}"
            partes.append(("+ " if partes else "") + texto)
        else:
            partes.append(_termino(c, nombre, not partes))
    return " ".join(partes)


def _ecuacion_homogenea(a, nombres):
    """Vector a como la ecuación a·x = 0: (1, -1, -1) -> 'x - y - z = 0'."""
    terminos = []
    for ai, v in zip(a, nombres):
        if ai != 0:
            terminos.append(_coeficiente(ai, v, not terminos))
    return f"{' '.join(terminos) if terminos else '0'} = 0"


def _subespacio(dim, n):
    """Nombre del subespacio según su dimensión: 'una recta por el origen', 'un plano por el origen', 'todo R³'..."""
    if dim == 0:
        return "{0} (solo el origen)"
    if dim == n:
        return f"todo {_R(n)}"
    if dim == 1:
        return "una recta por el origen"
    if dim == 2 and n == 3:
        return "un plano por el origen"
    return f"un subespacio de dimensión {dim} de {_R(n)} (pasa por el origen)"


def _tex_aumentada(M, corte, fmt=None):
    fmt = fmt or _latex
    columnas = "c" * corte + "|" + "c" * (M.cols - corte)
    filas = r" \\ ".join(" & ".join(fmt(M[i, j]) for j in range(M.cols)) for i in range(M.rows))
    return r"\left[\begin{array}{" + columnas + "}" + filas + r"\end{array}\right]"


def muestra_aumentada(nombre, M, corte, tex=None, n_cifras=4):
    """Muestra una matriz aumentada con la barra después de la columna 'corte' (tex: el nombre en LaTeX). Si la forma
    exacta es muy larga, solo los decimales, con el aviso."""
    tex_nombre = tex if tex is not None else _a_tex(nombre)
    if _exacta_legible(M):
        _emite(_Formula(f"{tex_nombre} = {_tex_aumentada(M, corte)}", f"{nombre} =\n{_plano(M, corte=corte)}"))
    else:
        dec = lambda e: cifras(e, n_cifras)
        _emite(_Formula(f"{tex_nombre} \\approx {_tex_aumentada(M, corte, dec)}" + r"\quad\text{(" + _LARGA + ")}",
                        f"{nombre} ≈\n{_plano(M, dec, corte=corte)}\n({_LARGA})"))


def _lista(cosas):
    """Números en una frase: [1, 2, 3] -> '1, 2 y 3'."""
    cosas = [str(c) for c in cosas]
    return cosas[0] if len(cosas) == 1 else ", ".join(cosas[:-1]) + " y " + cosas[-1]


def _coma(cosas):
    """Nombres separados con comas: ['v1', 'v2'] -> 'v1, v2'."""
    return ", ".join(str(c) for c in cosas)


def _flotante(M):
    """Matrix de sympy -> arreglo de numpy en punto flotante."""
    return np.array(sp.Matrix(M).evalf().tolist(), dtype=float)


def normaliza_texto(texto):
    """Minúsculas, sin acentos ni signos de puntuación y con un solo espacio entre palabras: '¿Dependientes?' -> 'dependientes'."""
    import unicodedata
    t = unicodedata.normalize("NFD", str(texto).lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    return " ".join(re.sub(r"[^a-z0-9ñ ]+", " ", t).split())


# Gráficas: un color, un estilo de línea y un marcador por vector, en orden fijo, para que se lean también en blanco y negro
_COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
_ESTILOS = ["-", "--", ":", "-.", (0, (5, 1, 1, 1.5, 1, 1.5)), (0, (1, 4))]          # continuo, discontinuo, punteado...
_MARCAS = ["o", "s", "^", "D", "v", "P"]


def _flecha(ax, origen, punta, color, estilo, ancho, texto=None, desplazamiento=(6, 6)):
    """Flecha en el plano de origen a punta; con texto, su rótulo junto a la punta (sobre un fondo blanco)."""
    if np.hypot(*(np.asarray(punta, float) - np.asarray(origen, float))) > 1e-12:
        ax.annotate("", xy=punta, xytext=origen,
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=ancho, linestyle=estilo, shrinkA=0, shrinkB=0))
    if texto:
        ax.annotate(texto, xy=punta, xytext=desplazamiento, textcoords="offset points", fontsize=11,
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85))


def _captura(fn, *args, **kwargs):
    """Ejecuta fn sin mostrar nada (ni texto ni gráficas) y devuelve (texto, error): lo que imprimió o mostró y la
    excepción que lanzó (None si no lanzó). Sirve para las pruebas de humo de las calculadoras."""
    partes, buf = [], io.StringIO()

    def vacia():
        if buf.getvalue():
            partes.append(("texto", "", buf.getvalue()))
            buf.seek(0); buf.truncate()

    mostrar, error = plt.show, None
    _BLOQUES.append((partes, vacia))
    plt.show = lambda *a, **k: plt.close("all")
    try:
        with contextlib.redirect_stdout(buf):
            fn(*args, **kwargs)
    except Exception as err:
        error = err
    finally:
        vacia()
        _BLOQUES.pop()
        plt.show = mostrar
        plt.close("all")
    return "\n".join(texto for _, _, texto in partes), error


def _humo(calculadora, controles, **cambios):
    """Prueba de humo: la calculadora corre con los valores de sus controles (y los cambios dados), no lanza ninguna
    excepción, escribe algo y no dice «Revisa la entrada»."""
    valores = {k: w.value for k, w in controles.items()}
    valores.update(cambios)
    texto, error = _captura(calculadora, **valores)
    return error is None and texto.strip() != "" and "Revisa la entrada" not in texto


def evalua(pruebas):
    """Ejecuta una vez cada caso (nombre, función que devuelve True o False). Devuelve [(nombre, ok)]; un caso
    que lanza una excepción cuenta como falla y su nombre dice por qué, sin interrumpir a los demás."""
    resultados = []
    for nombre, prueba in pruebas:
        try:
            ok = bool(prueba())
        except Exception as err:
            ok, nombre = False, f"{nombre}  (error: {type(err).__name__}: {err})"
        resultados.append((nombre, ok))
    return resultados


def corre_pruebas(pruebas, seccion):
    """Escribe OK o FALLA en cada caso y el resumen de la sección."""
    resultados = evalua(pruebas)
    for nombre, ok in resultados:
        print("OK  " if ok else "FALLA", nombre)
    print(f"Sección {seccion}: {sum(ok for _, ok in resultados)}/{len(pruebas)} casos pasan")

## 1. Subespacios de $\mathbb R^n$ y combinaciones lineales

**Combinación lineal.** Con vectores $\mathbf v_1,\dots,\mathbf v_k$ de $\mathbb R^n$ y números $c_1,\dots,c_k$, el vector $c_1\mathbf v_1+c_2\mathbf v_2+\dots+c_k\mathbf v_k$ es una *combinación lineal* de ellos, con *coeficientes* $c_1,\dots,c_k$.

**Generado.** $\operatorname{gen}\{\mathbf v_1,\dots,\mathbf v_k\}$ es el conjunto de **todas** las combinaciones lineales de esos vectores. Preguntar si $\mathbf b$ está en el generado es preguntar si el sistema

$$c_1\mathbf v_1+\dots+c_k\mathbf v_k=\mathbf b\qquad\Longleftrightarrow\qquad A\mathbf c=\mathbf b,\quad A=[\,\mathbf v_1\ \cdots\ \mathbf v_k\,]$$

tiene solución: los vectores son las **columnas** de $A$ y la respuesta la dan los rangos, $\operatorname{rango}A=\operatorname{rango}[A\,|\,\mathbf b]$ (unidad 3).

**Subespacio.** Un subconjunto $H$ de $\mathbb R^n$ es un *subespacio* si cumple las tres condiciones:

1. $\mathbf 0$ está en $H$;
2. $H$ es cerrado bajo la suma: si $\mathbf u$ y $\mathbf w$ están en $H$, $\mathbf u+\mathbf w$ también;
3. $H$ es cerrado bajo el producto por escalar: si $\mathbf u$ está en $H$ y $c$ es un número, $c\,\mathbf u$ también.

La condición 1 es la primera prueba: si un conjunto no contiene al origen, no es subespacio.

- **El generado siempre es un subespacio**: la suma de dos combinaciones de los $\mathbf v_i$ y un múltiplo de una combinación son otra vez combinaciones de los $\mathbf v_i$.
- $\operatorname{Nul}(A)=\{\mathbf x:\ A\mathbf x=\mathbf 0\}$ es un subespacio: si $A\mathbf x=\mathbf 0$ y $A\mathbf y=\mathbf 0$, entonces $A(\mathbf x+\mathbf y)=\mathbf 0$ y $A(c\,\mathbf x)=\mathbf 0$.
- $\operatorname{Col}(A)$, el generado de las columnas de $A$, es un subespacio (es un generado).
- Las soluciones de $A\mathbf x=\mathbf b$ con $\mathbf b\neq\mathbf 0$ **no** forman un subespacio: $\mathbf x=\mathbf 0$ da $A\mathbf 0=\mathbf 0\neq\mathbf b$, así que no contienen al origen.

**Cuáles hay.** En $\mathbb R^2$: $\{\mathbf 0\}$, las rectas por el origen y $\mathbb R^2$ completo. En $\mathbb R^3$: $\{\mathbf 0\}$, las rectas por el origen, los planos por el origen y $\mathbb R^3$ completo. Una recta o un plano que no pasa por el origen no es subespacio.

**Describir un generado con ecuaciones.** Un vector $\mathbf w$ está en $\operatorname{gen}\{\mathbf v_1,\dots,\mathbf v_k\}$ si y solo si cumple $\mathbf a\cdot\mathbf w=0$ para todo $\mathbf a$ con $\mathbf a\cdot\mathbf v_i=0$ para cada $i$, es decir, para todo $\mathbf a$ de $\operatorname{Nul}(A^T)$. Por eso las ecuaciones del generado se obtienen de una base de $\operatorname{Nul}(A^T)$: si las coordenadas de $\mathbf w$ son $(x,y,z)$, un plano por el origen en $\mathbb R^3$ tiene una ecuación $a x+b y+c z=0$, y una recta por el origen, dos. Si $\operatorname{rango}A=n$, el generado es todo $\mathbb R^n$ y no hay ecuaciones.

La calculadora escribe cada ecuación con coeficientes enteros sin factor común y con el primero distinto de cero positivo.

In [ ]:
def describe_generado(vectores):
    """Qué es gen{v1, ..., vk}: diccionario con dim (= rango de A), base (los vectores dados que son columnas pivote,
    numerados desde 1), ecuaciones (vectores a tales que a·x = 0 describe el generado: base de Nul(Aᵀ), escalada a
    enteros sin factor común y con la primera entrada distinta de cero positiva), es_todo (True si dim = n), n, k,
    la matriz A (los vectores como columnas) y su forma escalonada."""
    A = lista_vectores(vectores)
    n, k = A.shape
    el = _elimina(A, k, con_b=False)
    ecuaciones = [_primitivo(a) for a in base_nulo(A.T)]
    return {"dim": el["r"], "base": [c + 1 for c in el["pivotes"]], "ecuaciones": ecuaciones, "es_todo": el["r"] == n,
            "n": n, "k": k, "A": A, "escalonada": el["E"]}


def en_generado(vectores, b, pasos=False, n_cifras=4):
    """¿Está b en gen{v1, ..., vk}? Diccionario con es (True o False), r (rango de A), r_aum (rango de [A | b]),
    coeficientes (lista exacta c1, ..., ck con b = c1·v1 + ... + ck·vk; si hay infinitas combinaciones, en función
    de los parámetros s, t, ...), particular (la combinación con los parámetros en 0), parametros, las formas
    escalonada y reducida de [A | b] y, si no está, el renglón '0 = c' que lo prueba. Con pasos=True imprime el
    procedimiento."""
    A = lista_vectores(vectores)
    b = vector(b, "b")
    n, k = A.shape
    if b.rows != n:
        raise ValueError(f"b tiene {_componentes(b.rows)} y los vectores tienen {n}: deben tener el mismo número de componentes")
    el = _elimina(A.row_join(b), k, con_b=True)
    res = {"es": el["imposible"] is None, "r": el["r"], "r_aum": el["r_aum"], "A": A, "b": b,
           "escalonada": el["E"], "reducida": el["R"], "pivote": [c + 1 for c in el["pivotes"]],
           "coeficientes": None, "particular": None, "direcciones": [], "parametros": [], "renglon": None,
           "pasos": el["pasos"]}
    if el["imposible"] is not None:
        f, c = el["imposible"]
        res["renglon"] = f"R{f} de la forma escalonada de [A | b] dice 0 = {c}"
    else:
        x, xp, dirs, ps = solucion_general(el["R"], el["pivotes"], k)
        res.update(coeficientes=list(x), particular=list(xp), direcciones=dirs, parametros=ps)
    if pasos:
        _informe_en_generado(res, n_cifras, con_pasos=True)
    return res


def _nombra_vectores(A):
    print(("Vector: " if A.cols == 1 else "Vectores: ") + ", ".join(f"v{j + 1} = {_tupla(A[:, j])}" for j in range(A.cols)) +
          f"  ({A.cols} {'vector' if A.cols == 1 else 'vectores'} de {_R(A.rows)})")


def _tex_A(k):
    """LaTeX de A = [v1 v2 ... vk] con los vectores en negritas y separados."""
    return r"A=[\," + r"\ ".join(rf"\mathbf v_{{{j}}}" for j in range(1, k + 1)) + r"\,]"


def _nombre_A(k):
    return f"A = [{' '.join(f'v{j}' for j in range(1, k + 1))}]"


def _informe_generado(desc, n_cifras):
    A, n = desc["A"], desc["n"]
    _nombra_vectores(A)
    muestra(_nombre_A(A.cols), A, n_cifras, tex=_tex_A(A.cols))
    print(f"rango(A) = {desc['dim']} → dim gen = {desc['dim']}: el generado es {_subespacio(desc['dim'], n)}.")
    base, sobran = desc["base"], desc["k"] - len(desc["base"])
    if base:
        print(f"{'Columna pivote' if len(base) == 1 else 'Columnas pivote'}: {_lista(base)} → "
              f"{_lista([f'v{j}' for j in base])} {'basta' if len(base) == 1 else 'bastan'} para generarlo"
              + ("." if not sobran else
                 f"; {'el vector que sobra es combinación' if sobran == 1 else 'los que sobran son combinaciones'} "
                 f"de {'él' if len(base) == 1 else 'ellos'}."))
    else:
        print("Ninguna columna pivote: " + ("el vector es el cero." if desc["k"] == 1 else "todos los vectores son el cero."))
    if desc["es_todo"]:
        print(f"Como rango(A) = n = {n}, no hace falta ninguna ecuación: todo vector de {_R(n)} está en el generado.")
    else:
        nombres = nombres_incognitas(n)
        cuantas = len(desc["ecuaciones"])
        print(f"{'Ecuación' if cuantas == 1 else 'Ecuaciones'} del generado ({cuantas} = n - rango = {n} - {desc['dim']}; "
              f"de una base de Nul(Aᵀ)), en las coordenadas ({_coma(nombres)}):")
        for a in desc["ecuaciones"]:
            ceros = all((a.T * A[:, j])[0] == 0 for j in range(A.cols))
            todos = "la cumple v1" if A.cols == 1 else "la cumplen todos los v_i"
            print(f"  {_ecuacion_homogenea(a, nombres)}" + (f"   ({todos})" if ceros else "   (¡un v_i NO la cumple!)"))


def _informe_en_generado(res, n_cifras, con_pasos=False):
    A, b = res["A"], res["b"]
    n, k = A.shape
    if con_pasos:
        _nombra_vectores(A)
    print(f"¿b = {_tupla(b)} está en gen{{{_coma(f'v{j}' for j in range(1, k + 1))}}}? Se busca c con "
          f"{_texto_combinacion(sp.symbols(' '.join(f'c{j}' for j in range(1, k + 1)), seq=True))} = b, "
          "es decir, A c = b:")
    muestra_aumentada("[A | b]", A.row_join(b), k, tex=r"[A\,|\,\mathbf b]", n_cifras=n_cifras)
    if con_pasos:
        _imprime_pasos(_solo_gauss(res["pasos"]), corte=k, n_cifras=n_cifras)
    muestra_aumentada("forma escalonada de [A | b]", res["escalonada"], k,
                      tex=r"\text{forma escalonada de }[A\,|\,\mathbf b]", n_cifras=n_cifras)
    print(f"rango(A) = {res['r']},  rango[A | b] = {res['r_aum']}")
    if not res["es"]:
        print(f"rango[A | b] = rango(A) + 1: b NO está en el generado; {res['renglon']}, que ninguna combinación cumple.")
        return
    print("rango[A | b] = rango(A): b SÍ está en el generado.")
    muestra_aumentada("forma escalonada reducida", res["reducida"], k, n_cifras=n_cifras)
    c, ps = res["coeficientes"], res["parametros"]
    if ps:
        print(f"Hay infinitas combinaciones ({len(ps)} parámetro{'s' if len(ps) > 1 else ''}, {_coma(ps)}): "
              f"b = {_texto_combinacion(c, ps)}")
        print(f"Por ejemplo, con {'todos los parámetros en 0' if len(ps) > 1 else str(ps[0]) + ' = 0'}: "
              f"b = {_texto_combinacion(res['particular'])}")
    else:
        print(f"Una sola combinación: b = {_texto_combinacion(c)}   (coeficientes {_tupla(sp.Matrix(c), n_cifras)})")
    xp = sp.Matrix(res["particular"])
    print(f"Comprobación: A·{_tupla(xp)} = {_tupla(A * xp)}" + (" = b" if A * xp == b else "  ≠ b: ¡revisa!"))


def _base_ortonormal(V):
    """Base ortonormal (en punto flotante, solo para dibujar) de las columnas independientes de V."""
    Q = []
    for v in V.T:
        w = v - sum((q @ v) * q for q in Q)
        if np.linalg.norm(w) > 1e-9 * max(1.0, np.linalg.norm(v)):
            Q.append(w / np.linalg.norm(w))
    return Q


def _proyeccion_exacta(A, b):
    """Proyección ortogonal exacta de b sobre Col(A) (con las columnas pivote): solo para dibujar el segmento que
    separa a b del generado; la decisión de si b está o no la toman los rangos."""
    el = _elimina(A, A.cols, con_b=False)
    if not el["pivotes"]:
        return sp.zeros(A.rows, 1)
    B = A.extract(list(range(A.rows)), el["pivotes"])
    return B * (B.T * B).inv() * B.T * b


def _dentro_caja(p, d, L):
    """Intervalo de t en el que p + t·d queda dentro de la caja [-L, L]³ (vacío si t0 >= t1)."""
    t0, t1 = -np.inf, np.inf
    for k in range(3):
        if abs(d[k]) > 1e-12:
            a, c = sorted(((-L - p[k]) / d[k], (L - p[k]) / d[k]))
            t0, t1 = max(t0, a), min(t1, c)
        elif abs(p[k]) > L:
            return 0.0, 0.0
    return t0, t1


def _corte_con_caja(normal, L):
    """Vértices, en orden, del polígono en que el plano normal·x = 0 corta a la caja [-L, L]³ (para dibujarlo)."""
    esquinas = np.array([(x, y, z) for x in (-L, L) for y in (-L, L) for z in (-L, L)], dtype=float)
    puntos = []
    for i in range(8):
        for j in range(i + 1, 8):
            p, q = esquinas[i], esquinas[j]
            if np.count_nonzero(p != q) != 1:          # solo las 12 aristas de la caja
                continue
            fp, fq = normal @ p, normal @ q
            if fp == fq or fp * fq > 0:
                continue
            puntos.append(p + fp / (fp - fq) * (q - p))
    P = np.unique(np.round(np.array(puntos), 12), axis=0)
    centro = P.mean(axis=0)
    u = P[0] - centro
    u /= np.linalg.norm(u)
    w = np.cross(normal / np.linalg.norm(normal), u)
    angulos = np.arctan2((P - centro) @ w, (P - centro) @ u)
    return P[np.argsort(angulos)]


def dibuja_generado(vectores, b=None):
    """Dibuja gen{v1, ..., vk}: en R² la recta o el plano entero; en R³ la recta o el pedazo del plano que cabe en
    la caja, con los vectores dados y, si se da b, el punto b (círculo negro si está en el generado, cruz si no).
    Para n > 3 no dibuja y lo dice."""
    desc = describe_generado(vectores)
    A, n, dim = desc["A"], desc["n"], desc["dim"]
    res = en_generado(A, b) if b is not None else None
    if n > 3 or n == 1:
        print(f"No se dibuja: los vectores son de {_R(n)}" + (" y solo se pueden dibujar los de R² y R³." if n > 3 else
              "; en R¹ el generado es {0} o toda la recta real."))
        return desc
    V = _flotante(A)
    bf = _flotante(b if isinstance(b, sp.MatrixBase) else vector(b, "b")).ravel() if b is not None else None
    puntos = [np.zeros(n)] + [V[:, j] for j in range(A.cols)] + ([bf] if bf is not None else [])
    L = 1.25 * max(1.0, float(np.abs(np.vstack(puntos)).max()))
    Q = _base_ortonormal(_flotante(A.extract(list(range(n)), [j - 1 for j in desc["base"]]))) if dim else []
    titulo = f"gen{{{_coma(f'v{j}' for j in range(1, A.cols + 1))}}}: {_subespacio(dim, n)}"
    if dim == n == 3:
        titulo += " (el generado llena todo el espacio: no hay plano ni recta que dibujar)"
    if n == 2:
        fig, ax = plt.subplots(figsize=(5.8, 5.8))
        if dim == 2:
            ax.add_patch(plt.Rectangle((-L, -L), 2 * L, 2 * L, facecolor="0.85", edgecolor="0.5", hatch="//",
                                       alpha=0.5, label="el generado: todo el plano"))
        elif dim == 1:
            t = np.array([-2.0, 2.0]) * L
            ax.plot(t * Q[0][0], t * Q[0][1], "-.", color="0.4", lw=2.2, label="el generado: una recta por el origen")
        else:
            ax.plot(0, 0, "o", color="0.4", ms=10, label="el generado: solo el origen")
        for j in range(A.cols):
            _flecha(ax, (0, 0), V[:, j], _COLORES[j], _ESTILOS[j], 2, f"v{j + 1}")
        if bf is not None:
            if res["es"]:
                ax.plot(*bf, "o", color="black", ms=10, zorder=5, label=f"b = {_tupla(res['b'])}: está")
            else:
                ax.plot(*bf, "X", color="black", ms=12, zorder=5, label=f"b = {_tupla(res['b'])}: NO está")
        ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_aspect("equal")
        ax.axhline(0, color="black", lw=0.6); ax.axvline(0, color="black", lw=0.6)
        ax.grid(alpha=0.4); ax.set_xlabel("x"); ax.set_ylabel("y")
        lugar = "lower right"
    else:
        fig = plt.figure(figsize=(7, 6.5))
        ax = fig.add_subplot(projection="3d")
        if dim == 2:
            ax.add_collection3d(Poly3DCollection([_corte_con_caja(np.cross(Q[0], Q[1]), L)], facecolor="0.6",
                                                 alpha=0.22, edgecolor="0.4", linewidth=1.2))
            rotulo = "el generado: un plano por el origen"
            for c in np.linspace(-1.6 * L, 1.6 * L, 9):                    # rejilla del plano, recortada a la caja
                for d, o in ((Q[0], Q[1]), (Q[1], Q[0])):
                    t0, t1 = _dentro_caja(c * o, d, L)
                    if t0 < t1:
                        P = np.outer([t0, t1], d) + c * o
                        ax.plot(*P.T, "-", color="0.4", lw=0.8, label=rotulo)
                        rotulo = None                                   # una sola entrada en la leyenda
        elif dim == 1:
            t = np.array([-1.0, 1.0]) * L / float(np.max(np.abs(Q[0])))       # la recta, de pared a pared de la caja
            P = np.outer(t, Q[0])
            ax.plot(*P.T, "-.", color="0.4", lw=2.2, label="el generado: una recta por el origen")
        elif dim == 0:
            ax.scatter(0, 0, 0, s=60, color="0.4", depthshade=False, label="el generado: solo el origen")
        for j in range(A.cols):
            ax.plot([0, V[0, j]], [0, V[1, j]], [0, V[2, j]], linestyle=_ESTILOS[j], color=_COLORES[j], lw=2.2,
                    label=f"v{j + 1} = {_tupla(A[:, j])}")
            ax.scatter(*V[:, j], marker=_MARCAS[j], s=45, color=_COLORES[j], depthshade=False)
        if bf is not None:
            if res["es"]:
                ax.scatter(*bf, s=90, marker="o", color="black", depthshade=False, label=f"b = {_tupla(res['b'])}: está")
            else:
                ax.scatter(*bf, s=110, marker="X", color="black", depthshade=False, label=f"b = {_tupla(res['b'])}: NO está")
                if dim > 0:
                    p = _flotante(_proyeccion_exacta(A, res["b"])).ravel()
                    ax.plot(*np.array([bf, p]).T, ":", color="black", lw=1.8, label="distancia de b al generado")
        ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_zlim(-L, L)
        ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
        ax.set_box_aspect((1, 1, 1))                         # misma escala en los tres ejes
        ax.view_init(elev=22, azim=-55)
        lugar = "upper left"
    if res is not None:
        titulo += " · b " + ("está" if res["es"] else "NO está")
    ax.set_title(textwrap.fill(titulo, 60), fontsize=10)
    ax.legend(loc=lugar, fontsize=8)
    plt.show()
    return desc

**Recorrido paso a paso.** ¿Los vectores $(5,4,1)$ y $(5,4,3)$ están en $\operatorname{gen}\{(1,0,1),(2,1,1)\}$? Se escalona $[A\,|\,\mathbf b]$ con los dos vectores como columnas de $A$. En el primer caso los rangos coinciden y sale la combinación; en el segundo aparece un renglón $0=c$. Al final, la descripción del generado (un plano por el origen y su ecuación) y su dibujo con los dos puntos.

In [ ]:
_r1 = en_generado("1, 0, 1; 2, 1, 1", "5, 4, 1", pasos=True)
print("\n" + "=" * 90 + "\n")
_r2 = en_generado("1, 0, 1; 2, 1, 1", "5, 4, 3", pasos=True)
print("\n" + "=" * 90 + "\n")
_informe_generado(describe_generado("1, 0, 1; 2, 1, 1"), 4)
dibuja_generado("1, 0, 1; 2, 1, 1", "5, 4, 1")
dibuja_generado("1, 0, 1; 2, 1, 1", "5, 4, 3");

**Calculadora.** Escribe los vectores **uno por renglón** (cada renglón es un vector y se vuelve una columna de $A$) y, si quieres, el vector $\mathbf b$; déjalo vacío para solo describir el generado. Prueba también con vectores de $\mathbb R^2$, por ejemplo `1, 2` y `2, 4` (una recta), o con más vectores que componentes.

In [ ]:
def calculadora_generado(vectores_txt, b_txt, dibujar, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; el dibujo va después
        try:
            A = lista_vectores(vectores_txt)
            b = vector(b_txt, "b") if str(b_txt).strip() else None
            desc = describe_generado(A)
            res = en_generado(A, b) if b is not None else None
        except ValueError as err:
            print("Revisa la entrada:", err); return
        _informe_generado(desc, n_cifras)
        if res is not None:
            print()
            _informe_en_generado(res, n_cifras)
    if dibujar:
        dibuja_generado(A, b)


_CONTROLES_1 = dict(
    vectores_txt=widgets.Textarea(value="1, 0, 1\n2, 1, 1", description="vectores:", continuous_update=False,
                                  layout=widgets.Layout(width="340px", height="120px")),
    b_txt=widgets.Text(value="5, 4, 1", description="b =", continuous_update=False),
    dibujar=widgets.Checkbox(value=True, description="dibujar (R² y R³)"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_generado, **_CONTROLES_1);

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
PRUEBAS_1 = [
    ("v1 = (1, 0, 1), v2 = (2, 1, 1), b = (5, 4, 1) → está, coeficientes (-3, 4), sin parámetros",
     lambda: (lambda r: r["es"] and r["coeficientes"] == [-3, 4] and r["parametros"] == [])(
         en_generado("1,0,1; 2,1,1", "5,4,1"))),
    ("mismos vectores, b = (5, 4, 3) → no está, r = 2, r_aum = 3",
     lambda: (lambda r: not r["es"] and (r["r"], r["r_aum"]) == (2, 3) and r["coeficientes"] is None)(
         en_generado("1,0,1; 2,1,1", "5,4,3"))),
    ("describe_generado((1, 0, 1), (2, 1, 1)) → dim 2, base [1, 2], ecuación (1, -1, -1): x - y - z = 0",
     lambda: (lambda d: d["dim"] == 2 and d["base"] == [1, 2] and d["ecuaciones"] == [vector("1, -1, -1")]
                        and not d["es_todo"])(describe_generado("1,0,1; 2,1,1"))),
    ("describe_generado((1, 2), (2, 4)) → dim 1, base [1], ecuación (2, -1): 2x - y = 0",
     lambda: (lambda d: d["dim"] == 1 and d["base"] == [1] and d["ecuaciones"] == [vector("2, -1")])(
         describe_generado("1,2; 2,4"))),
    ("describe_generado((2, 1, 0), (-1, 0, 1)) → dim 2, ecuación (1, -2, 1)",
     lambda: (lambda d: d["dim"] == 2 and d["ecuaciones"] == [vector("1, -2, 1")])(describe_generado("2,1,0; -1,0,1"))),
    ("describe_generado((1, 0), (1, 1)) → dim 2, es todo R², sin ecuaciones",
     lambda: (lambda d: d["dim"] == 2 and d["es_todo"] and d["ecuaciones"] == [])(describe_generado("1,0; 1,1"))),
    ("b = 0 siempre está: coeficientes (0, 0)",
     lambda: (lambda r: r["es"] and r["coeficientes"] == [0, 0])(en_generado("1,0,1; 2,1,1", "0,0,0"))),
    ("e1, e2, e3 y (1, 1, 1) con b = (2, 3, 4) → está, con 1 parámetro, y toda la familia da b",
     lambda: (lambda r: r["es"] and len(r["parametros"]) == 1
                        and sp.expand(r["A"] * sp.Matrix(r["coeficientes"])) == vector("2, 3, 4"))(
         en_generado("1,0,0; 0,1,0; 0,0,1; 1,1,1", "2,3,4"))),
    ("vectores de distinto tamaño (1, 2) y (1, 2, 3) → ValueError que lo explica",
     lambda: "mismo número de componentes" in _mensaje(lambda: en_generado("1,2; 1,2,3", "1,2"))
             and "vector 2" in _mensaje(lambda: describe_generado("1,2; 1,2,3"))),
    ("prueba de humo: la calculadora corre con sus valores iniciales, sin error y sin «Revisa la entrada»",
     lambda: _humo(calculadora_generado, _CONTROLES_1)),
]
corre_pruebas(PRUEBAS_1, 1)

**Contrasta (sección 1).** ¿El vector $(1,2,3)$ está en $\operatorname{gen}\{(1,1,1),(0,1,2)\}$? Plantea $c_1(1,1,1)+c_2(0,1,2)=(1,2,3)$ y resuélvelo a mano. Si está, escribe los coeficientes `mis_coeficientes = (c1, c2)`; si no, escribe `mis_coeficientes = 'no'`.

In [ ]:
mis_coeficientes = None     # si está: dos números, por ejemplo (4, -5) o "1/2, 3"; si no está: 'no'

if mis_coeficientes is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    respuesta = None
    if isinstance(mis_coeficientes, str) and normaliza_texto(mis_coeficientes) in (
            "no", "no esta", "no pertenece", "no esta en el generado"):
        respuesta = "no"
    else:
        try:
            respuesta = vector(mis_coeficientes, "tus coeficientes")
        except ValueError as err:
            print(f"Revisa tu respuesta: {str(err).rstrip('.')} (o escribe «no» si el vector no está en el generado).")
        else:
            if respuesta.rows != 2:          # respuesta imposible: la referencia todavía no se muestra
                print(f"Son dos coeficientes, c1 y c2 (escribiste {respuesta.rows}): revisa tu respuesta y vuelve a "
                      "ejecutar la celda.")
                respuesta = None
    if respuesta is not None:
        ref = en_generado("1, 1, 1; 0, 1, 2", "1, 2, 3")
        if ref["es"]:
            print("La calculadora da: sí está, con coeficientes", _tupla(sp.Matrix(ref["coeficientes"])) + ".")
        else:
            print(f"La calculadora da: no está ({ref['renglon']}).")
        if (respuesta == "no") == (not ref["es"]) and (respuesta == "no" or respuesta == sp.Matrix(ref["coeficientes"])):
            print("Coinciden.")
        elif respuesta == "no" or not ref["es"]:
            print("NO coinciden: escalona [A | b] con (1, 1, 1) y (0, 1, 2) como columnas; si no aparece un renglón "
                  "0 = c con c distinto de 0, hay combinación.")
        else:
            print("NO coinciden: la primera componente da c1 directo; sustituye en la segunda para c2 y comprueba "
                  "en la tercera.")
        print("\nEl procedimiento completo:")
        en_generado("1, 1, 1; 0, 1, 2", "1, 2, 3", pasos=True)

## 2. Independencia lineal

**Definición.** Los vectores $\mathbf v_1,\dots,\mathbf v_k$ son *linealmente independientes* si la única manera de obtener el vector cero,

$$c_1\mathbf v_1+c_2\mathbf v_2+\dots+c_k\mathbf v_k=\mathbf 0,$$

es con **todos** los $c_i=0$. Si hay una combinación con algún $c_i\neq 0$ que da $\mathbf 0$ (una *relación de dependencia*), son *linealmente dependientes*.

- **Dependencia ⇔ alguno sobra.** Son dependientes si y solo si alguno de ellos es combinación lineal de los otros: si $c_j\neq 0$ en la relación, despejas $\mathbf v_j$. Quitar ese vector no cambia el generado.
- **Prueba con el rango.** Con $A=[\,\mathbf v_1\ \cdots\ \mathbf v_k\,]$, la ecuación de arriba es el sistema homogéneo $A\mathbf c=\mathbf 0$. Por eso

  $$\text{independientes}\iff\operatorname{rango}A=k\iff\operatorname{Nul}(A)=\{\mathbf 0\}.$$

  Cada columna sin pivote es un vector que se escribe con los de las columnas pivote, y cada vector de una base de $\operatorname{Nul}(A)$ es una relación de dependencia.
- **$n$ vectores en $\mathbb R^n$.** $A$ es cuadrada: son independientes si y solo si $\det A\neq 0$.
- **Más de $n$ vectores en $\mathbb R^n$** siempre son dependientes: el rango no pasa de $n$, que es menor que $k$.
- **Un conjunto que contiene al $\mathbf 0$** es dependiente: $1\cdot\mathbf 0=\mathbf 0$ es una relación con un coeficiente distinto de cero.
- **Dos vectores** son dependientes si y solo si son paralelos (uno es múltiplo del otro; el $\mathbf 0$ cuenta como múltiplo de cualquiera). **Con tres o más no basta revisar de dos en dos:** $(1,2,0)$, $(0,1,3)$ y $(2,5,3)$ no tienen ningún par paralelo y aun así son dependientes, porque el tercero está en el plano de los dos primeros.

La calculadora escribe cada relación con coeficientes enteros sin factor común y con el primero distinto de cero positivo, por ejemplo `2·v1 + 1·v2 - 1·v3 = 0`.

In [ ]:
def independencia(vectores):
    """¿Son linealmente independientes? Diccionario con rango, k (número de vectores), n (componentes),
    independientes (True o False), relaciones (base de Nul(A) escalada a enteros sin factor común: cada una es una
    relación de dependencia), textos de las relaciones ('2·v1 + 1·v2 - 1·v3 = 0'), pivote (columnas pivote, desde
    1), redundantes ({j: [(i, c), ...]}: el vector j, que no es pivote, como combinación exacta de los pivote),
    pares_paralelos (pares (i, j) paralelos o con un cero), det (si k = n; None si no) y las formas de A."""
    A = lista_vectores(vectores)
    n, k = A.shape
    el = _elimina(A, k, con_b=False)
    piv = el["pivotes"]
    relaciones = [_primitivo(d) for d in _direcciones(el["R"], piv, k)]
    redundantes = {j + 1: [(c + 1, el["R"][f, j]) for f, c in enumerate(piv)] for j in el["libres"]}
    pares = [(i + 1, j + 1) for i in range(k) for j in range(i + 1, k)
             if sp.Matrix.hstack(A[:, i], A[:, j]).rank() < 2]
    return {"rango": el["r"], "k": k, "n": n, "independientes": el["r"] == k, "relaciones": relaciones,
            "textos": [texto_relacion(c) for c in relaciones], "pivote": [c + 1 for c in piv],
            "redundantes": redundantes, "pares_paralelos": pares, "det": A.det() if k == n else None,
            "A": A, "escalonada": el["E"], "reducida": el["R"], "pasos": el["pasos"]}


def _texto_redundante(j, pares, n_cifras=None):
    """'v3 = 2·v1 + 1·v2' a partir de [(1, 2), (2, 1)]; con n_cifras, los coeficientes no enteros también en decimales."""
    if not pares:
        return f"v{j} = 0 (es el vector cero)"
    coef = [c for _, c in pares]
    texto = f"v{j} = " + _texto_combinacion(coef, nombres=[f"v{i}" for i, _ in pares])
    if n_cifras is not None and not all(c.is_integer for c in coef):
        texto += f"   (coeficientes ≈ {_aprox(coef, n_cifras)})"
    return texto


def _informe_independencia(res, n_cifras, con_pasos=False):
    A, n, k, r = res["A"], res["n"], res["k"], res["rango"]
    uno = k == 1
    _nombra_vectores(A)
    muestra(_nombre_A(k), A, n_cifras, tex=_tex_A(k))
    if con_pasos:
        _imprime_pasos(_solo_gauss(res["pasos"]), n_cifras=n_cifras)
    muestra("forma escalonada", res["escalonada"], n_cifras)
    print(f"rango(A) = {r},  k = {k} {'vector' if uno else 'vectores'} en {_R(n)}")
    if res["det"] is not None and not uno:
        d = res["det"]
        igual = _igual("det A", d, n_cifras)
        print(f"Como k = n = {n}, A es cuadrada: {igual}" +
              ((" ≠ 0" if "(" not in igual else ", distinto de 0,") + " → independientes." if d != 0 else " → dependientes."))
    if k > n:
        print(f"Hay más vectores ({k}) que componentes ({n}): el rango no pasa de {n}, así que tienen que ser dependientes.")
    ceros = [j + 1 for j in range(k) if A[:, j] == sp.zeros(n, 1)]
    if ceros:
        print(("El vector es el cero: es dependiente (1·0 = 0)." if uno else
               f"El conjunto contiene al vector cero ({_lista([f'v{j}' for j in ceros])}): es dependiente."))
    if res["independientes"]:
        if uno:
            print("rango = k = 1: el vector es LINEALMENTE INDEPENDIENTE (no es el cero); Nul(A) = {0}.")
        else:
            print(f"rango = k = {k}: los vectores son LINEALMENTE INDEPENDIENTES; Nul(A) = {{0}}: la única combinación "
                  "que da 0 es la que tiene todos los coeficientes en 0.")
        return
    print(f"rango = {r} < k = {k}: " + ("el vector es LINEALMENTE DEPENDIENTE." if uno else
                                        "los vectores son LINEALMENTE DEPENDIENTES."))
    piv = res["pivote"]
    if piv:
        print(f"{'Columna pivote' if len(piv) == 1 else 'Columnas pivote'}: {_lista(piv)}; cada vector que no es pivote "
              f"se escribe con {'el vector pivote' if len(piv) == 1 else 'los vectores pivote'}:")
        for j, pares in res["redundantes"].items():
            print(f"  {_texto_redundante(j, pares, n_cifras)}")
    else:
        print("Ninguna columna pivote: " + ("el vector es el cero." if uno else "todos los vectores son el cero."))
    dim = len(res["relaciones"])
    print(f"{'Relación' if dim == 1 else 'Relaciones'} de dependencia (base de Nul(A), dim {dim} = k - rango = {k} - {r}):")
    for c, texto in zip(res["relaciones"], res["textos"]):
        print(f"  {texto}" + ("   (comprobado: A·c = 0)" if A * c == sp.zeros(n, 1) else "   (¡A·c ≠ 0!)"))
    if res["pares_paralelos"]:
        print(f"{'Par paralelo' if len(res['pares_paralelos']) == 1 else 'Pares paralelos'} (uno es múltiplo del otro, "
              "o uno es el cero): " + ", ".join(f"(v{i}, v{j})" for i, j in res["pares_paralelos"]))
    elif k >= 2:
        print("Pares paralelos: ninguno.")
        print("Ningún par es paralelo y aun así son dependientes: con tres o más vectores no basta revisar de dos en dos.")

**Recorrido paso a paso.** Los vectores $(1,2,0)$, $(0,1,3)$ y $(2,5,3)$. Ningún par es paralelo, pero al escalonar $A$ la tercera columna se queda sin pivote: el rango es 2 y aparece la relación $2\mathbf v_1+\mathbf v_2-\mathbf v_3=\mathbf 0$, es decir, $\mathbf v_3=2\mathbf v_1+\mathbf v_2$. El tercero está en el plano de los dos primeros.

In [ ]:
_informe_independencia(independencia("1, 2, 0; 0, 1, 3; 2, 5, 3"), 4, con_pasos=True)

**Calculadora.** Escribe los vectores **uno por renglón**. Prueba un conjunto con el vector cero, dos vectores paralelos, más vectores que componentes, y tres vectores de $\mathbb R^3$ sin pares paralelos que sí sean independientes.

In [ ]:
def calculadora_independencia(vectores_txt, n_cifras):
    with bloque():                 # el resultado aparece completo, de una sola vez
        try:
            res = independencia(vectores_txt)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        _informe_independencia(res, n_cifras)


_CONTROLES_2 = dict(
    vectores_txt=widgets.Textarea(value="1, 2, 0\n0, 1, 3\n2, 5, 3", description="vectores:", continuous_update=False,
                                  layout=widgets.Layout(width="340px", height="130px")),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_independencia, **_CONTROLES_2);

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
PRUEBAS_2 = [
    ("(2, -1), (-6, 3) → dependientes; relación 3·v1 + 1·v2 = 0; el par (1, 2) es paralelo",
     lambda: (lambda r: not r["independientes"] and r["textos"] == ["3·v1 + 1·v2 = 0"]
                        and r["pares_paralelos"] == [(1, 2)])(independencia("2,-1; -6,3"))),
    ("(1, 0, 1), (0, 1, 1), (1, 1, 0) → independientes, det = -2",
     lambda: (lambda r: r["independientes"] and r["det"] == -2 and r["relaciones"] == [])(independencia("1,0,1; 0,1,1; 1,1,0"))),
    ("(1, 2, 0), (0, 1, 3), (2, 5, 3) → dependientes, relación (2, 1, -1), ningún par paralelo",
     lambda: (lambda r: not r["independientes"] and r["relaciones"] == [vector("2, 1, -1")]
                        and r["textos"] == ["2·v1 + 1·v2 - 1·v3 = 0"] and r["pares_paralelos"] == []
                        and r["redundantes"] == {3: [(1, 2), (2, 1)]})(independencia("1,2,0; 0,1,3; 2,5,3"))),
    ("e1, e2, e3, (1, 1, 1) → dependientes, rango 3, relación (1, 1, 1, -1)",
     lambda: (lambda r: not r["independientes"] and r["rango"] == 3 and r["relaciones"] == [vector("1, 1, 1, -1")])(
         independencia("1,0,0; 0,1,0; 0,0,1; 1,1,1"))),
    ("(1, 2, 3), (0, 0, 0) → dependientes (contiene al cero)",
     lambda: (lambda r: not r["independientes"] and r["rango"] == 1 and (1, 2) in r["pares_paralelos"])(
         independencia("1,2,3; 0,0,0"))),
    ("un solo vector no nulo, (4, -2, 7) → independiente",
     lambda: (lambda r: r["independientes"] and r["rango"] == 1 and r["k"] == 1)(independencia("4,-2,7"))),
    ("corrientes (10, -5, -5), (5, 5, -10), (-5, 10, -5) → dependientes, relación 1·v1 - 1·v2 + 1·v3 = 0",
     lambda: (lambda r: not r["independientes"] and r["relaciones"] == [vector("1, -1, 1")] and r["det"] == 0
                        and r["textos"] == ["1·v1 - 1·v2 + 1·v3 = 0"])(independencia("10,-5,-5; 5,5,-10; -5,10,-5"))),
    ("(1, 0, 0, 1), (0, 1, 0, 1), (0, 0, 1, 1) → independientes, rango 3",
     lambda: (lambda r: r["independientes"] and r["rango"] == 3 and r["det"] is None)(
         independencia("1,0,0,1; 0,1,0,1; 0,0,1,1"))),
    ("prueba de humo: la calculadora corre con sus valores iniciales, sin error y sin «Revisa la entrada»",
     lambda: _humo(calculadora_independencia, _CONTROLES_2)),
]
corre_pruebas(PRUEBAS_2, 2)

**Contrasta (sección 2).** ¿Los vectores $(1,0,2)$, $(0,1,1)$ y $(2,1,5)$ son linealmente independientes? Escalona a mano $A=[\,\mathbf v_1\ \mathbf v_2\ \mathbf v_3\,]$ (los vectores como columnas), cuenta los pivotes y escribe `mi_rango = ` (un entero) y `mi_respuesta = 'independientes'` o `'dependientes'`.

In [ ]:
mi_rango = None         # un entero, por ejemplo 3
mi_respuesta = None     # 'independientes' o 'dependientes' (también 'LI', 'L.D.', 'no son independientes', ...)

# Respuestas que se aceptan, ya normalizadas (minúsculas, sin acentos ni puntuación). Se compara la frase completa.
_RESPUESTAS = {
    "independientes": ["independientes", "independiente", "linealmente independientes", "linealmente independiente",
                       "son independientes", "es independiente", "son linealmente independientes",
                       "es linealmente independiente", "li", "l i", "no son dependientes", "no son linealmente dependientes",
                       "no dependientes", "no son ld"],
    "dependientes": ["dependientes", "dependiente", "linealmente dependientes", "linealmente dependiente",
                     "son dependientes", "es dependiente", "son linealmente dependientes", "es linealmente dependiente",
                     "ld", "l d", "no son independientes", "no son linealmente independientes", "no independientes",
                     "no son li"],
}


def _respuesta_independencia(texto):
    t = normaliza_texto(texto)
    for clave, frases in _RESPUESTAS.items():
        if t in frases:
            return clave
    raise ValueError(f"«{texto}» no es una de las respuestas que reconoce esta celda: escribe «independientes» "
                     "o «dependientes»")


if mi_rango is None or mi_respuesta is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escribe tu rango y tu respuesta arriba y vuelve a ejecutar la celda.")
else:
    try:
        mio = numero(mi_rango, "tu rango")
        mia = _respuesta_independencia(mi_respuesta)
    except ValueError as err:
        print(f"Revisa tu respuesta: {str(err).rstrip('.')}.")
    else:
        if not (mio.is_integer and 0 <= mio <= 3):     # respuesta imposible: la referencia todavía no se muestra
            print(f"El rango de tres vectores de R³ es un entero de 0 a 3 (escribiste {mio}): revisa tu número "
                  "y vuelve a ejecutar la celda.")
        elif (mio == 3) != (mia == "independientes"):  # incoherente: tampoco se muestra la referencia
            print(f"Tu rango ({mio}) y tu respuesta ({mia}) no son congruentes: con tres vectores, son independientes "
                  "si y solo si el rango es 3. Revisa cuál de los dos está mal y vuelve a ejecutar la celda.")
        else:
            ref = independencia("1, 0, 2; 0, 1, 1; 2, 1, 5")
            clave = "independientes" if ref["independientes"] else "dependientes"
            print(f"La calculadora da: rango {ref['rango']}, {clave}" +
                  (f" ({ref['textos'][0]})" if ref["textos"] else "") + ".")
            print("Rango:", "coincide." if mio == ref["rango"] else
                  "NO coincide: escalona con los vectores como COLUMNAS y cuenta los renglones que no quedan en ceros.")
            print("Respuesta:", "coincide." if mia == clave else
                  "NO coincide: son independientes solo si el rango es igual al número de vectores (3).")
            print("\nEl procedimiento completo:")
            _informe_independencia(ref, 4, con_pasos=True)

## 3. Base, dimensión, coordenadas y cambio de base

**Base.** Una *base* de un subespacio $H$ es un conjunto de vectores de $H$ que es **linealmente independiente** y **genera** $H$. Todas las bases de $H$ tienen el mismo número de vectores: la *dimensión* de $H$, $\dim H$ ($\dim\{\mathbf 0\}=0$).

- La **base estándar** de $\mathbb R^n$ es $\mathbf e_1=(1,0,\dots,0),\ \dots,\ \mathbf e_n=(0,\dots,0,1)$, así que $\dim\mathbb R^n=n$.
- En $\mathbb R^n$, **$n$ vectores independientes ya son base** (generan todo $\mathbb R^n$); con $n$ vectores, eso equivale a $\det[\,\mathbf b_1\ \cdots\ \mathbf b_n\,]\neq 0$.

**Los tres subespacios de una matriz $A$ de $m\times n$.** Al escalonar $A$:

- **base de $\operatorname{Col}(A)$**: las columnas pivote **de la $A$ original** (no las de la forma reducida: escalonar cambia el espacio columna);
- **base de $\operatorname{Nul}(A)$**: los vectores de la solución general de $A\mathbf x=\mathbf 0$, uno por variable libre;
- **base del espacio renglón $\operatorname{Ren}(A)$**: los renglones no nulos de la forma escalonada (las operaciones de renglón no cambian el espacio renglón).

$$\operatorname{rango}(A)=\dim\operatorname{Col}(A)=\dim\operatorname{Ren}(A),\qquad \operatorname{rango}(A)+\dim\operatorname{Nul}(A)=n\quad\text{(teorema de la dimensión)}.$$

**Coordenadas.** Si $B=\{\mathbf b_1,\dots,\mathbf b_n\}$ es base, todo $\mathbf x$ se escribe **de una sola manera** como $\mathbf x=c_1\mathbf b_1+\dots+c_n\mathbf b_n$ (de dos maneras, la resta daría una relación de dependencia). Los coeficientes son las *coordenadas de $\mathbf x$ en la base $B$*, $[\mathbf x]_B=(c_1,\dots,c_n)$. Con $P_B=[\,\mathbf b_1\ \cdots\ \mathbf b_n\,]$:

$$\mathbf x=P_B\,[\mathbf x]_B,\qquad [\mathbf x]_B=P_B^{-1}\,\mathbf x.$$

**Cambio de base.** Si $B$ y $C$ son bases de $\mathbb R^n$, de $P_B[\mathbf x]_B=\mathbf x=P_C[\mathbf x]_C$ sale

$$[\mathbf x]_C=P_{C\leftarrow B}\,[\mathbf x]_B,\qquad P_{C\leftarrow B}=P_C^{-1}P_B .$$

Las columnas de $P_{C\leftarrow B}$ son las coordenadas en $C$ de los vectores de $B$.

**Base de un subespacio.** Si $B$ tiene $k<n$ vectores independientes, es base del subespacio que generan (un plano en $\mathbb R^3$, por ejemplo). Un $\mathbf x$ de ese subespacio tiene $k$ coordenadas; un $\mathbf x$ fuera de él no tiene coordenadas en $B$.

In [ ]:
def _revisa_base(P, nombre="la base"):
    """ValueError si las columnas de P (los vectores de la base) no son independientes."""
    n, k = P.shape
    if k > n:
        raise ValueError(f"no es base: {nombre} tiene {k} vectores en {_R(n)}, y más de {n} vectores en {_R(n)} "
                         "siempre son dependientes")
    r = _elimina(P, k, con_b=False)["r"]
    if r < k:
        detalle = f" (det = {P.det()})" if k == n else f" (rango {r} < {k} vectores)"
        raise ValueError(f"no es base: los vectores son dependientes{detalle}")


def coordenadas(base, x):
    """[x]_B exacto: los coeficientes c con x = c1·b1 + ... + ck·bk (B escrita como lista de vectores, uno por
    renglón). Si B tiene n vectores dependientes en Rⁿ: ValueError 'no es base: los vectores son dependientes'.
    Si B tiene k < n vectores independientes (base de un subespacio), da las coordenadas si x está en el
    subespacio; si no, ValueError 'x no está en el subespacio generado por la base'."""
    P = lista_vectores(base, "la base")
    x = vector(x, "x")
    n, k = P.shape
    if x.rows != n:
        raise ValueError(f"x tiene {_componentes(x.rows)} y los vectores de la base tienen {n}: deben coincidir")
    _revisa_base(P)
    el = _elimina(P.row_join(x), k, con_b=True)
    if el["imposible"] is not None:
        raise ValueError("x no está en el subespacio generado por la base")
    return el["R"][:k, k]


def desde_coordenadas(base, c):
    """x = P_B·[x]_B = c1·b1 + ... + ck·bk."""
    P = lista_vectores(base, "la base")
    c = vector(c, "las coordenadas")
    if c.rows != P.cols:
        raise ValueError(f"escribiste {_componentes(c.rows).replace('componente', 'coordenada')} y la base tiene "
                         f"{P.cols} {'vector' if P.cols == 1 else 'vectores'}: hace falta una coordenada por vector")
    _revisa_base(P)
    return P * c


def matriz_cambio(B, C):
    """P_{C←B} = P_C⁻¹·P_B, exacta (B y C, bases de Rⁿ escritas como listas de vectores)."""
    PB, PC = lista_vectores(B, "la base B"), lista_vectores(C, "la base C")
    if PB.rows != PC.rows:
        raise ValueError(f"los vectores de B tienen {PB.rows} componentes y los de C {PC.rows}: deben ser del mismo espacio")
    for P, letra in ((PB, "B"), (PC, "C")):
        n, k = P.shape
        if k != n:
            raise ValueError(f"la base {letra} tiene {k} {'vector' if k == 1 else 'vectores'} en {_R(n)}: para cambiar de "
                             f"base en {_R(n)}, cada base necesita {n} vectores")
        try:
            _revisa_base(P, f"la base {letra}")
        except ValueError as err:
            raise ValueError(f"{letra} {err}") from None
    return PC.inv() * PB


def base_col_nul(A):
    """Bases de Col(A), Nul(A) y Ren(A) de una matriz A escrita POR RENGLONES (hasta 6×6). Diccionario con reducida,
    escalonada, pivote (columnas pivote, desde 1), base_col (columnas pivote de la A ORIGINAL), base_nul (vectores
    exactos de la solución general de A x = 0), base_ren (renglones no nulos de la forma escalonada, como vectores),
    rango, nulidad, n y comprobacion (rango + nulidad == n)."""
    A = matriz_limitada(A, "A")
    m, n = A.shape
    el = _elimina(A, n, con_b=False)
    piv = el["pivotes"]
    base_nul = _direcciones(el["R"], piv, n)
    return {"A": A, "escalonada": el["E"], "reducida": el["R"], "pivote": [c + 1 for c in piv],
            "base_col": [A[:, c] for c in piv], "base_nul": base_nul,
            "base_ren": [el["E"].row(i).T for i in range(el["r"])], "rango": el["r"], "nulidad": len(base_nul),
            "n": n, "comprobacion": el["r"] + len(base_nul) == n}


def _informe_coordenadas(P, x, c, n_cifras, modo="x → coordenadas", letra="b"):
    """Informe de las coordenadas; letra es el nombre de los vectores de la base (b1, b2... o a1, a2...)."""
    n, k = P.shape
    nombres = [f"{letra}{j}" for j in range(1, k + 1)]
    print("Base B: " + ", ".join(f"{nm} = {_tupla(P[:, j])}" for j, nm in enumerate(nombres)))
    muestra("P_B", P, n_cifras)
    if k == n:
        d = P.det()
        igual = _igual("det P_B", d, n_cifras)
        print(igual + (" ≠ 0: " if "(" not in igual else ", distinto de 0: ") +
              (f"el vector es independiente, así que B es base de {_R(n)}." if n == 1 else
              f"los {n} vectores son independientes, así que B es base de {_R(n)}."))
    else:
        print(f"B tiene {k} {'vector' if k == 1 else 'vectores'} {'independiente' if k == 1 else 'independientes'} en "
              f"{_R(n)}: es base de {_subespacio(k, n)}, no de todo {_R(n)}.")
    if modo == "x → coordenadas":
        if k == n:
            muestra("P_B⁻¹", P.inv(), n_cifras)
            print(f"[x]_B = P_B⁻¹·x = {_tupla(c, n_cifras)}")
        else:
            print(f"[x]_B sale de resolver P_B c = x (los rangos coinciden: x está en el subespacio): {_tupla(c, n_cifras)}")
    else:
        print(f"x = P_B·[x]_B = {_tupla(x, n_cifras)}")
    print(f"Comprobación: {_texto_combinacion(list(c), nombres=nombres)} = {_tupla(P * c)}" +
          (" = x" if P * c == x else "  ≠ x: ¡revisa!"))


_TOPE_DIBUJO = 10 ** 6          # más allá de esto las rejillas ya no se pueden leer: no se dibujan


def _paso_rejilla(lo, hi, maximo=40):
    """Paso entero para trazar a lo más unas 40 líneas entre lo y hi."""
    return max(1, math.ceil((hi - lo) / maximo))


def dibuja_rejillas(base, x, letra="b"):
    """Solo R²: rejilla estándar (gris claro, continua) y rejilla de la base B (discontinua), los vectores de la base
    (el primero continuo, el segundo discontinuo), el punto x y el camino c1·b1 seguido de c2·b2 (punteado grueso);
    el título da las coordenadas en las dos bases. Cada rejilla lleva a lo más unas 40 líneas; si las coordenadas
    pasan de 10⁶, no dibuja y lo dice."""
    P = lista_vectores(base, "la base")
    xv = vector(x, "x")
    if P.shape != (2, 2):
        raise ValueError(f"las rejillas se dibujan para una base de R² con 2 vectores (esta tiene {P.cols} "
                         f"{'vector' if P.cols == 1 else 'vectores'} de {P.rows} componentes)")
    c = coordenadas(P, xv)
    b1, b2 = _flotante(P[:, 0]).ravel(), _flotante(P[:, 1]).ravel()
    xf, cf = _flotante(xv).ravel(), _flotante(c).ravel()
    puntos = np.vstack([np.zeros(2), b1, b2, xf, cf[0] * b1])
    L = 1.25 * max(1.0, float(np.abs(puntos).max())) + 0.5
    if not math.isfinite(L) or L > _TOPE_DIBUJO or float(np.abs(cf).max()) > _TOPE_DIBUJO:
        print(f"No se dibujan las rejillas: las coordenadas pasan de {_TOPE_DIBUJO:,} y la figura no se podría leer.")
        return c
    n1, n2 = f"{letra}1", f"{letra}2"
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    K = math.ceil(L)
    paso = _paso_rejilla(-K, K)
    for k in range(-K - (-K % paso), K + 1, paso):          # rejilla estándar (a lo más unas 40 líneas)
        ax.axvline(k, color="0.82", lw=0.8, zorder=0, label=None)
        ax.axhline(k, color="0.82", lw=0.8, zorder=0)
    ax.plot([], [], "-", color="0.82", lw=0.8, label="rejilla estándar" + (f" (cada {paso})" if paso > 1 else ""))
    esquinas = np.array([[-L, -L], [L, -L], [L, L], [-L, L]])
    en_B = (_flotante(P.inv()) @ esquinas.T).T                  # esquinas de la ventana en coordenadas de B (P_B⁻¹ exacta)
    pasos_B = []
    for eje, (d, otro) in enumerate(((b2, b1), (b1, b2))):      # líneas i·b1 + t·b2 y j·b2 + t·b1
        lo, hi = math.floor(en_B[:, eje].min()), math.ceil(en_B[:, eje].max())
        paso_B = _paso_rejilla(lo, hi)
        pasos_B.append(paso_B)
        t = np.array([-1.0, 1.0]) * 4 * L / max(np.hypot(*d), 1e-12)
        for i in range(lo - (lo % paso_B), hi + 1, paso_B):
            base_pt = i * otro
            ax.plot(base_pt[0] + t * d[0], base_pt[1] + t * d[1], "--", color=_COLORES[0], lw=0.9, alpha=0.8)
    ax.plot([], [], "--", color=_COLORES[0], lw=0.9, label="rejilla de B" + (" (no todas las líneas)" if max(pasos_B) > 1 else ""))
    _flecha(ax, (0, 0), b1, _COLORES[1], "-", 2.6, n1, (6, -14))
    _flecha(ax, (0, 0), b2, _COLORES[2], "--", 2.6, n2)
    ax.plot([], [], "-", color=_COLORES[1], lw=2.6, label=f"{n1} (flecha continua)")
    ax.plot([], [], "--", color=_COLORES[2], lw=2.6, label=f"{n2} (flecha discontinua)")
    p1 = cf[0] * b1
    camino = f"camino: {_termino(c[0], n1, True)} y luego {_termino(c[1], n2, True)}"
    ax.plot(*np.array([[0, 0], p1]).T, ":", color="black", lw=3, label=camino)
    ax.plot(*np.array([p1, xf]).T, ":", color="black", lw=3)
    ax.plot(*p1, "s", color="black", ms=6)
    ax.plot(*xf, "o", color="black", ms=9, zorder=5, label=f"x = {_tupla(xv)}")
    ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_aspect("equal")
    ax.axhline(0, color="black", lw=0.8); ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("x (estándar)"); ax.set_ylabel("y (estándar)")
    ax.set_title(textwrap.fill(f"En la base estándar: x = {_tupla(xv)};  en la base B: [x]_B = {_tupla(c)}", 70), fontsize=10)
    ax.legend(loc="lower right", fontsize=8)
    plt.show()
    return c

**Recorrido paso a paso.** La semilla del libro: en una ciudad las calles van **al este**, $\mathbf a_1=(1,0)$, y **al noreste**, $\mathbf a_2=(1,1)$; la base es $B=\{\mathbf a_1,\mathbf a_2\}$. Para llegar a $\mathbf x=(3,2)$ (3 al este y 2 al norte en el plano usual) caminas $c_1$ cuadras al este y $c_2$ al noreste: $c_1(1,0)+c_2(1,1)=(3,2)$, así que $c_2=2$ y $c_1=1$, es decir, $[\mathbf x]_B=(1,2)$. En la gráfica, la rejilla de $B$ (discontinua) es el plano de calles: $\mathbf x$ queda en la esquina 1 al este y 2 al noreste.

In [ ]:
_PB = lista_vectores("1, 0; 1, 1", "la base")
_xB = coordenadas(_PB, "3, 2")
_informe_coordenadas(_PB, vector("3, 2"), _xB, 4, letra="a")
dibuja_rejillas("1, 0; 1, 1", "3, 2", letra="a");

**Calculadora de coordenadas.** Escribe los vectores de la base $B$ **uno por renglón** y un vector. En el modo «x → coordenadas», el vector es $\mathbf x$ y la calculadora da $[\mathbf x]_B$; en el modo «coordenadas → x», el vector es $[\mathbf x]_B$ y la calculadora da $\mathbf x$. Con una base de $\mathbb R^2$ dibuja las dos rejillas. Prueba una base que no lo sea (`1, 2; 2, 4`) y una base de un plano de $\mathbb R^3$ con un $\mathbf x$ fuera de él.

In [ ]:
def calculadora_coordenadas(base_txt, vector_txt, modo, dibujar, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; el dibujo va después
        try:
            P = lista_vectores(base_txt, "la base")
            if modo == "x → coordenadas":
                x = vector(vector_txt, "x")
                c = coordenadas(P, x)
            else:
                c = vector(vector_txt, "[x]_B")
                x = desde_coordenadas(P, c)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        _informe_coordenadas(P, x, c, n_cifras, modo)
    if dibujar:
        if P.shape == (2, 2):
            dibuja_rejillas(P, x)
        else:
            print("Las rejillas se dibujan solo para una base de R² (dos vectores de 2 componentes).")


_CONTROLES_3A = dict(
    base_txt=widgets.Textarea(value="1, 0\n1, 1", description="base B:", continuous_update=False,
                              layout=widgets.Layout(width="300px", height="100px")),
    vector_txt=widgets.Text(value="3, 2", description="x o [x]_B:", continuous_update=False,
                            style={"description_width": "initial"}),
    modo=widgets.Dropdown(options=["x → coordenadas", "coordenadas → x"], value="x → coordenadas", description="modo:"),
    dibujar=widgets.Checkbox(value=True, description="dibujar rejillas (R²)"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_coordenadas, **_CONTROLES_3A);

**Calculadora de los subespacios de una matriz.** Escribe $A$ **por renglones** (como en la unidad 3). La calculadora da las bases de $\operatorname{Col}(A)$, $\operatorname{Nul}(A)$ y $\operatorname{Ren}(A)$, el rango, la nulidad y comprueba el teorema de la dimensión.

In [ ]:
def calculadora_col_nul(A_txt, n_cifras):
    with bloque():
        try:
            res = base_col_nul(A_txt)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        A, n = res["A"], res["n"]
        muestra("A", A, n_cifras)
        muestra("forma escalonada", res["escalonada"], n_cifras)
        muestra("forma escalonada reducida", res["reducida"], n_cifras)
        piv = res["pivote"]
        print(f"{'Columna pivote' if len(piv) == 1 else 'Columnas pivote'}: {_lista(piv) if piv else 'ninguna'}")
        if piv:
            print(f"Base de Col(A) ({'columna' if len(piv) == 1 else 'columnas'} {_lista(piv)} de la A ORIGINAL): " +
                  ", ".join(_tupla(v, n_cifras) for v in res["base_col"]))
        else:
            print("Base de Col(A): ninguna, porque A no tiene columnas pivote: Col(A) = {0}")
        print("Base de Nul(A) (solución general de A x = 0, un vector por variable libre): " +
              (", ".join(_tupla(v, n_cifras) for v in res["base_nul"]) if res["base_nul"] else "ninguna: Nul(A) = {0}"))
        print("Base de Ren(A) (renglones no nulos de la forma escalonada): " +
              (", ".join(_tupla(v, n_cifras) for v in res["base_ren"]) if res["base_ren"] else "ninguna: Ren(A) = {0}"))
        print(f"rango(A) = dim Col(A) = dim Ren(A) = {res['rango']};  nulidad = dim Nul(A) = {res['nulidad']}")
        print(f"Teorema de la dimensión: rango + nulidad = {res['rango']} + {res['nulidad']} = "
              f"{res['rango'] + res['nulidad']} = n (columnas de A) = {n}" + ("  ✓" if res["comprobacion"] else "  ✗"))
        ceros = all(A * v == sp.zeros(A.rows, 1) for v in res["base_nul"])
        if res["base_nul"]:
            print("Comprobación: A·v = 0 para cada vector de la base de Nul(A)." if ceros else "Comprobación FALLÓ: A·v ≠ 0.")


_CONTROLES_3B = dict(
    A_txt=widgets.Textarea(value="1, 2, 3; 2, 4, 7; 3, 6, 10", description="A =", continuous_update=False,
                           layout=widgets.Layout(width="340px", height="110px")),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_col_nul, **_CONTROLES_3B);

**Calculadora de cambio de base.** Escribe las bases $B$ y $C$ de $\mathbb R^n$ (los vectores separados por `;`) y unas coordenadas $[\mathbf x]_B$. La calculadora arma $P_{C\leftarrow B}=P_C^{-1}P_B$ y da $[\mathbf x]_C$.

In [ ]:
def calculadora_cambio(B_txt, C_txt, xB_txt, n_cifras):
    with bloque():
        try:
            M = matriz_cambio(B_txt, C_txt)
            PB, PC = lista_vectores(B_txt, "la base B"), lista_vectores(C_txt, "la base C")
            xB = vector(xB_txt, "[x]_B")
            if xB.rows != PB.cols:
                raise ValueError(f"[x]_B tiene {_componentes(xB.rows)} y la base B tiene {PB.cols} vectores: "
                                 "hace falta una coordenada por vector")
        except ValueError as err:
            print("Revisa la entrada:", err); return
        muestra("P_B", PB, n_cifras)
        muestra("P_C", PC, n_cifras)
        muestra("P_C⁻¹", PC.inv(), n_cifras)
        muestra("P_{C←B} = P_C⁻¹·P_B", M, n_cifras)
        xC = M * xB
        x = PB * xB
        print(f"[x]_C = P_{{C←B}}·[x]_B = {_tupla(xC, n_cifras)}")
        print(f"Comprobación: x = P_B·[x]_B = {_tupla(x)} y P_C·[x]_C = {_tupla(PC * xC)}" +
              ("  (el mismo vector)" if PC * xC == x else "  ¡no coinciden!"))


_CONTROLES_3C = dict(
    B_txt=widgets.Text(value="1, 0; 1, 1", description="B:", continuous_update=False),
    C_txt=widgets.Text(value="1, 2; -1, 1", description="C:", continuous_update=False),
    xB_txt=widgets.Text(value="1, 2", description="[x]_B =", continuous_update=False),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_cambio, **_CONTROLES_3C);

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
PRUEBAS_3 = [
    ("base (1, 0), (1, 1), x = (3, 2) → [x]_B = (1, 2)",
     lambda: coordenadas("1,0; 1,1", "3,2") == vector("1, 2")),
    ("base (1, 2), (3, 4), x = (5, 6) → [x]_B = (-1, 2)",
     lambda: coordenadas("1,2; 3,4", "5,6") == vector("-1, 2")),
    ("(1, 2), (2, 4) no es base → ValueError «no es base: los vectores son dependientes»",
     lambda: _mensaje(lambda: coordenadas("1,2; 2,4", "1,1")).startswith("no es base: los vectores son dependientes")),
    ("P_(C←B) con B = (1, 0), (1, 1) y C = (1, 1), (1, -1) → [[1/2, 1], [1/2, 0]]; aplicada a (1, 2) da (5/2, 1/2)",
     lambda: (lambda M: M == sp.Matrix([[sp.Rational(1, 2), 1], [sp.Rational(1, 2), 0]])
                        and M * vector("1, 2") == vector("5/2, 1/2"))(matriz_cambio("1,0; 1,1", "1,1; 1,-1"))),
    ("base_col_nul(1, 2, 3; 2, 4, 7; 3, 6, 10) → pivote [1, 3], Col: (1, 2, 3), (3, 7, 10), Nul: (-2, 1, 0), 2 + 1 = 3",
     lambda: (lambda r: r["pivote"] == [1, 3] and r["base_col"] == [vector("1, 2, 3"), vector("3, 7, 10")]
                        and len(r["base_nul"]) == 1 and sp.Matrix.hstack(r["base_nul"][0], vector("-2, 1, 0")).rank() == 1
                        and (r["rango"], r["nulidad"], r["n"]) == (2, 1, 3) and r["comprobacion"])(
         base_col_nul("1,2,3; 2,4,7; 3,6,10"))),
    ("base de un plano (2, 1, 0), (-1, 0, 1): x = (3, 2, 1) → (2, 1); x = (1, 0, 0) → no está en el subespacio",
     lambda: coordenadas("2,1,0; -1,0,1", "3,2,1") == vector("2, 1")
             and _mensaje(lambda: coordenadas("2,1,0; -1,0,1", "1,0,0")) == "x no está en el subespacio generado por la base"),
    ("base (1/2, 1/2), (1/2, -1/2), x = (30, 10) → [x]_B = (40, 20)",
     lambda: coordenadas("1/2,1/2; 1/2,-1/2", "30,10") == vector("40, 20")),
    ("ida y vuelta con la base (1, 2), (3, 4) y x = (7, -3): desde_coordenadas(B, coordenadas(B, x)) = x",
     lambda: desde_coordenadas("1,2; 3,4", coordenadas("1,2; 3,4", "7,-3")) == vector("7, -3")),
    ("base_col_nul de la identidad 3×3 → rango 3, base de Nul vacía",
     lambda: (lambda r: r["rango"] == 3 and r["base_nul"] == [] and r["comprobacion"])(base_col_nul(sp.eye(3)))),
    ("base de Ren(1, 2, 3; 2, 4, 7; 3, 6, 10) → los dos renglones no nulos de la escalonada: (1, 2, 3) y (0, 0, 1)",
     lambda: base_col_nul("1,2,3; 2,4,7; 3,6,10")["base_ren"] == [vector("1, 2, 3"), vector("0, 0, 1")]),
    ("desde_coordenadas con una base dependiente, (1, 2), (2, 4) → ValueError «no es base»",
     lambda: _mensaje(lambda: desde_coordenadas("1,2; 2,4", "1, 1")).startswith("no es base")),
    ("prueba de humo: la calculadora de coordenadas corre con sus valores iniciales, en los dos modos",
     lambda: _humo(calculadora_coordenadas, _CONTROLES_3A)
             and _humo(calculadora_coordenadas, _CONTROLES_3A, vector_txt="1, 2", modo="coordenadas → x")),
    ("prueba de humo: la calculadora de Col, Nul y Ren corre con sus valores iniciales",
     lambda: _humo(calculadora_col_nul, _CONTROLES_3B)),
    ("prueba de humo: la calculadora de cambio de base corre con sus valores iniciales",
     lambda: _humo(calculadora_cambio, _CONTROLES_3C)),
    ("rejillas con coordenadas enormes: no se cuelga (x = (10000, 1) con b2 = (1, 0.0001); x = (1e15, 0))",
     lambda: (lambda t0: _humo(calculadora_coordenadas, _CONTROLES_3A, base_txt="1, 0; 1, 0.0001", vector_txt="10000, 1")
                         and "No se dibujan" in _captura(calculadora_coordenadas, "1, 0; 1, 1", "1e15, 0",
                                                         "x → coordenadas", True, 4)[0]
                         and time.time() - t0 < 20)(time.time())),
]
corre_pruebas(PRUEBAS_3, 3)

**Contrasta (sección 3).** Calcula a mano las coordenadas de $\mathbf x=(4,1)$ en la base $B=\{(1,1),(1,-1)\}$: plantea $c_1(1,1)+c_2(1,-1)=(4,1)$ y resuelve. Escribe `mis_coordenadas = (c1, c2)`; puedes usar fracciones como texto (`"7/2"`) o decimales.

In [ ]:
mis_coordenadas = None     # dos números, por ejemplo (3, -1/2) escrito como ("3", "-1/2") o (3, -0.5)

if mis_coordenadas is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mias = vector(mis_coordenadas, "tus coordenadas")
    except ValueError as err:
        print(f"Revisa tus coordenadas: {str(err).rstrip('.')}.")
    else:
        if mias.rows != 2:          # respuesta imposible: la referencia todavía no se muestra
            print(f"La base tiene dos vectores, así que son dos coordenadas (escribiste {mias.rows}): revisa tu "
                  "respuesta y vuelve a ejecutar la celda.")
        else:
            ref = coordenadas("1, 1; 1, -1", "4, 1")
            print(f"La calculadora da: [x]_B = {_tupla(ref, 4)}.")
            if mias == ref:
                print("Coinciden.")
            elif all(abs(float(a - b)) <= 0.005 for a, b in zip(mias, ref)):
                print("Coinciden salvo redondeo (escribe la fracción exacta si puedes).")
            else:
                print("NO coinciden: suma y resta las dos ecuaciones, c1 + c2 = 4 y c1 - c2 = 1; comprueba que "
                      "c1·(1, 1) + c2·(1, -1) dé (4, 1).")
            print("\nEl procedimiento completo:")
            _P = lista_vectores("1, 1; 1, -1", "la base")
            _informe_coordenadas(_P, vector("4, 1"), ref, 4)

## 4. Transformaciones lineales: rotación, escalamiento y proyección

**Transformación lineal.** Una función $T:\mathbb R^n\to\mathbb R^m$ es *lineal* si, para todos los vectores $\mathbf u,\mathbf v$ y todo número $c$,

$$T(\mathbf u+\mathbf v)=T(\mathbf u)+T(\mathbf v)\qquad\text{y}\qquad T(c\,\mathbf u)=c\,T(\mathbf u).$$

Entonces $T(\mathbf 0)=T(0\cdot\mathbf 0)=0\cdot T(\mathbf 0)=\mathbf 0$, y vale la **superposición**: $T(c_1\mathbf u_1+\dots+c_k\mathbf u_k)=c_1T(\mathbf u_1)+\dots+c_kT(\mathbf u_k)$.

**Matriz estándar.** Como $\mathbf x=x_1\mathbf e_1+\dots+x_n\mathbf e_n$, la superposición da $T(\mathbf x)=x_1T(\mathbf e_1)+\dots+x_nT(\mathbf e_n)=A\mathbf x$ con

$$A=[\,T(\mathbf e_1)\ \ T(\mathbf e_2)\ \cdots\ T(\mathbf e_n)\,]:$$

basta saber a dónde van los vectores de la base estándar.

**Las del plano y el espacio.**

- **Rotación** un ángulo $\theta$ en sentido antihorario: $R(\theta)=\begin{bmatrix}\cos\theta&-\operatorname{sen}\theta\\ \operatorname{sen}\theta&\cos\theta\end{bmatrix}$. Ojo con la unidad: $\cos 60^\circ=\tfrac12$, pero $\cos(60\ \text{rad})\approx-0.952$. Por eso `rotacion(angulo, unidad)` te pide **siempre** la unidad, `"grados"` o `"radianes"`. En radianes puedes escribir el ángulo con pi: `"pi/6"`, `"2*pi/3"` o `"π/4"`.
- **Rotaciones en $\mathbb R^3$** alrededor de un eje (antihorarias vistas desde la punta del eje, regla de la mano derecha):

  $$R_x(\theta)=\begin{bmatrix}1&0&0\\0&\cos\theta&-\operatorname{sen}\theta\\0&\operatorname{sen}\theta&\cos\theta\end{bmatrix},\quad R_y(\theta)=\begin{bmatrix}\cos\theta&0&\operatorname{sen}\theta\\0&1&0\\-\operatorname{sen}\theta&0&\cos\theta\end{bmatrix},\quad R_z(\theta)=\begin{bmatrix}\cos\theta&-\operatorname{sen}\theta&0\\ \operatorname{sen}\theta&\cos\theta&0\\0&0&1\end{bmatrix}.$$
- **Escalamiento:** $\operatorname{diag}(k_1,k_2)=\begin{bmatrix}k_1&0\\0&k_2\end{bmatrix}$ estira (o encoge) $k_1$ veces en $x$ y $k_2$ veces en $y$.
- **Proyección** sobre la recta por el origen de dirección $\mathbf u\neq\mathbf 0$: $P=\dfrac{\mathbf u\,\mathbf u^T}{\mathbf u\cdot\mathbf u}$, porque $P\mathbf x=\dfrac{\mathbf u\cdot\mathbf x}{\mathbf u\cdot\mathbf u}\,\mathbf u$. Cumple $P^2=P$ (proyectar dos veces es proyectar una).
- **Reflexión** respecto a esa recta: $H=2P-I$ (el punto medio entre $\mathbf x$ y $H\mathbf x$ es $P\mathbf x$). Cumple $H^2=I$. En el plano, $\det H=-1$; en $\mathbb R^n$, $\det H=(-1)^{n-1}$, porque la recta queda fija y las $n-1$ direcciones perpendiculares a ella cambian de sentido.
- **Composición:** aplicar $T_1$ y después $T_2$ tiene matriz $A_2A_1$: **la que actúa primero va a la derecha**. En general $A_2A_1\neq A_1A_2$.

**Núcleo e imagen.** $\operatorname{Nul}(T)=\{\mathbf x:\ T(\mathbf x)=\mathbf 0\}=\operatorname{Nul}(A)$ son los vectores que se aplastan en el origen; la imagen es $\operatorname{Col}(A)$, todo lo que $T$ alcanza. Por el teorema de la dimensión, $\dim\operatorname{Nul}(T)+\dim\operatorname{Im}(T)=n$. Una proyección del plano tiene núcleo de dimensión 1 (la recta perpendicular a $\mathbf u$) e imagen de dimensión 1 (la recta de $\mathbf u$); una rotación tiene núcleo $\{\mathbf 0\}$ e imagen $\mathbb R^2$.

**No toda función "lineal" lo es.** Una traslación $\mathbf x\mapsto\mathbf x+\mathbf b$ con $\mathbf b\neq\mathbf 0$ y la función $y=mx+b$ de la prepa con $b\neq 0$ **no** son transformaciones lineales: no mandan $\mathbf 0$ a $\mathbf 0$. Por ejemplo, un transmisor de temperatura con salida de 4 a 20 mA, $I=4+0.16\,T$, no es lineal en este sentido (es *afín*), aunque su gráfica sea una recta.

In [ ]:
_UNIDADES = {"grados": "grados", "grado": "grados", "radianes": "radianes", "radian": "radianes", "rad": "radianes"}


def _unidad(unidad):
    """'grados' o 'radianes' (acepta mayúsculas, acentos, el singular y 'rad'). No hay unidad por omisión: sin unidad,
    o con otra cosa, ValueError en español."""
    if unidad is None or (isinstance(unidad, str) and not unidad.strip()):
        raise ValueError("indica la unidad: «grados» o «radianes» (no hay unidad por omisión)")
    t = normaliza_texto(unidad) if isinstance(unidad, str) else ""
    if t not in _UNIDADES:
        raise ValueError(f"la unidad del ángulo debe ser «grados» o «radianes» (escribiste «{unidad}»)")
    return _UNIDADES[t]


_AYUDA_ANGULO = ("escribe un número (30, 37.5, 1/3) o, en radianes, un múltiplo de pi (pi/6, 2*pi/3, π/4, -pi)")


def angulo_exacto(angulo):
    """El ángulo como número exacto: 30 -> 30, '37.5' -> 75/2, 'pi/6' -> π/6, '2*pi/3', '2π/3', '-π/4'. El lector es
    restringido: solo números, pi, productos y cocientes (no evalúa expresiones libres)."""
    if not isinstance(angulo, str):
        return numero(angulo, "el ángulo")
    t = _normaliza_entrada(angulo).replace(" ", "").lower().replace("π", "pi")
    if "pi" not in t:
        return numero(angulo, "el ángulo")
    signo = 1
    if t[:1] in "+-":
        signo, t = (-1 if t[0] == "-" else 1), t[1:]
    t = re.sub(r"(?<=[\d.])pi", "*pi", t)                     # 2pi -> 2*pi
    trozos = re.split(r"([*/])", t)
    if not t or "" in trozos[::2]:
        raise ValueError(f"«{angulo}» no es un ángulo que entienda esta calculadora: {_AYUDA_ANGULO}")
    coef, potencia, op = sp.Integer(1), 0, "*"
    for i, trozo in enumerate(trozos):
        if i % 2:
            op = trozo
            continue
        if trozo == "pi":
            potencia += 1 if op == "*" else -1
            continue
        try:
            q = numero(trozo, "el ángulo")
        except ValueError:
            raise ValueError(f"«{angulo}» no es un ángulo que entienda esta calculadora: {_AYUDA_ANGULO}") from None
        if op == "/" and q == 0:
            raise ValueError(f"«{angulo}» divide entre cero")
        coef = coef * q if op == "*" else coef / q
    if potencia not in (0, 1):
        raise ValueError(f"«{angulo}» no es un ángulo que entienda esta calculadora: {_AYUDA_ANGULO}")
    return signo * coef * sp.pi ** potencia


def _angulo_rad(angulo, unidad):
    """El ángulo exacto en radianes: 30 grados -> pi/6; 30 radianes -> 30; 'pi/6' radianes -> pi/6."""
    u = _unidad(unidad)
    a = angulo_exacto(angulo)
    if u == "grados":
        if a.has(sp.pi):
            raise ValueError("con la unidad «grados» escribe el ángulo sin pi (por ejemplo 30); pi/6 ya está en radianes")
        return sp.rad(a)
    return a


def _num_angulo(a):
    """El número de un ángulo como texto: 30 -> '30', 75/2 -> '37.5', 1/3 -> '1/3', pi/6 -> 'π/6', 2*pi/3 -> '2π/3'."""
    if a.has(sp.pi):
        return _txt(a).replace("·π", "π")
    if a.is_integer:
        return str(a)
    q = int(a.q)
    while q % 2 == 0:
        q //= 2
    while q % 5 == 0:
        q //= 5
    return repr(float(a)) if q == 1 else str(a)          # decimal finito (37.5) o fracción (1/3)


def _texto_angulo(angulo, unidad):
    """'30 grados', '1 grado', 'π/6 radianes', '1 radián'."""
    a, u = angulo_exacto(angulo), _unidad(unidad)
    uno = abs(a) == 1
    nombre = ("grado" if uno else "grados") if u == "grados" else ("radián" if uno else "radianes")
    return f"{_num_angulo(a)} {nombre}"


def _tex_angulo(angulo, unidad):
    a = angulo_exacto(angulo)
    return rf"{_latex(a)}^\circ" if _unidad(unidad) == "grados" else rf"{_latex(a)}\ \text{{rad}}"


def rotacion(angulo, unidad=None):
    """R(θ) exacta, antihoraria. La unidad es obligatoria: rotacion(30, "grados"), rotacion("pi/6", "radianes");
    sin unidad, ValueError «indica la unidad»."""
    t = _angulo_rad(angulo, unidad)
    c, s = sp.cos(t), sp.sin(t)
    return sp.Matrix([[c, -s], [s, c]])


def rotacion3(eje, angulo, unidad=None):
    """Rotación exacta de R³ alrededor del eje 'x', 'y' o 'z' (antihoraria vista desde la punta del eje, regla de la
    mano derecha). La unidad es obligatoria, como en rotacion."""
    e = normaliza_texto(eje) if isinstance(eje, str) else ""
    if e not in ("x", "y", "z"):
        raise ValueError(f"el eje de rotación debe ser «x», «y» o «z» (escribiste «{eje}»)")
    t = _angulo_rad(angulo, unidad)
    c, s = sp.cos(t), sp.sin(t)
    if e == "x":
        return sp.Matrix([[1, 0, 0], [0, c, -s], [0, s, c]])
    if e == "y":
        return sp.Matrix([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    return sp.Matrix([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def escalamiento(*k):
    """diag(k1, k2, ...) exacta: escalamiento(1.5, 0.5) -> diag(3/2, 1/2)."""
    if not k:
        raise ValueError("escribe al menos un factor de escala, por ejemplo escalamiento(2, 1)")
    if len(k) > 6:
        raise ValueError(f"escribiste {len(k)} factores; esta calculadora acepta hasta 6")
    return sp.diag(*[numero(v, f"el factor k{i}") for i, v in enumerate(k, 1)])


def _direccion(u):
    """La dirección u de una recta: un vector no nulo de hasta 6 componentes."""
    u = vector(u, "la dirección u")
    if u.rows > 6:
        raise ValueError(f"la dirección u tiene {u.rows} componentes; esta calculadora acepta hasta 6")
    if u == sp.zeros(u.rows, 1):
        raise ValueError("la dirección u no puede ser el vector cero: el 0 no define ninguna recta")
    return u


def proyeccion(u):
    """Proyección exacta sobre la recta por el origen de dirección u: P = u uᵀ / (u·u)."""
    u = _direccion(u)
    return u * u.T / (u.T * u)[0]


def reflexion(u):
    """Reflexión exacta respecto a la recta por el origen de dirección u: H = 2P - I."""
    P = proyeccion(u)
    return 2 * P - sp.eye(P.rows)


def matriz_desde_imagenes(imagenes):
    """Matriz estándar A = [T(e1) ... T(en)] a partir de las imágenes de la base estándar, escritas una por renglón
    (como lista de vectores): '2, 1; -1, 3' son T(e1) = (2, 1) y T(e2) = (-1, 3), y quedan como columnas."""
    return lista_vectores(imagenes, "la lista de imágenes T(e1), ..., T(en)")


def _matriz_t(A, nombre="A"):
    """Matriz de una transformación: texto POR RENGLONES (racional, hasta 6×6) o Matrix exacta de sympy, que puede
    tener raíces, senos y cosenos (la de una rotación)."""
    if isinstance(A, sp.MatrixBase) and not all(sp.sympify(e).is_Rational for e in A):
        M = sp.Matrix(A)
        _revisa_tamano(M, nombre)
        if not all(e.is_number and e.is_real for e in M):
            raise ValueError(f"{nombre} debe tener entradas numéricas reales")
        return M
    return matriz_limitada(A, nombre)


def nucleo_imagen(A):
    """Núcleo e imagen de T(x) = A x: diccionario con rango, base_nucleo (base exacta de Nul(A)), base_imagen
    (columnas pivote de la A ORIGINAL), dim_nucleo, dim_imagen, n (columnas: T va de Rⁿ...) y m (renglones: ...a Rᵐ)."""
    A = _matriz_t(A)
    m, n = A.shape
    el = _elimina(A, n, con_b=False)
    piv = el["pivotes"]
    nucleo = _direcciones(el["R"], piv, n)
    return {"A": A, "rango": el["r"], "base_nucleo": nucleo, "base_imagen": [A[:, c] for c in piv],
            "dim_nucleo": len(nucleo), "dim_imagen": el["r"], "n": n, "m": m, "pivote": [c + 1 for c in piv]}


def _iguales(a, b):
    """True si los vectores a y b son iguales: exacto con fracciones, raíces, senos...; con decimales de punto
    flotante (si la función los usa), con una tolerancia relativa de 1e-9."""
    if a.shape != b.shape:
        return False
    for p, q in zip(a, b):
        d = sp.sympify(p - q)
        if d.has(sp.Float):
            if abs(float(d)) > 1e-9 * max(1.0, abs(float(p)), abs(float(q))):
                return False
        elif not _es_cero(d):
            return False
    return True


_SIN_CONTRAEJEMPLO = ("no se encontró contraejemplo con estas pruebas (es una prueba con vectores fijos, no una "
                      "demostración); si T es lineal, su matriz estándar es A = [T(e1) ... T(en)]")


def prueba_linealidad(f, n):
    """Busca un contraejemplo a la linealidad de f: Rⁿ → Rᵐ. f recibe una Matrix columna de sympy de tamaño n y
    devuelve otra (o una lista). Revisa primero T(0) = 0; después la aditividad T(u + v) = T(u) + T(v) y la
    homogeneidad T(c·u) = c·T(u) con vectores fijos, enteros y no enteros (e1, ..., en, (1, 2, ..., n),
    (3, -1, 4, ...), (1/2, -1/3, 2/5, ...)) y los escalares 2, -3, 1/2 y -1/3, en forma exacta; al final, que
    T(u) = A·u con A = [T(e1) ... T(en)]. Devuelve un diccionario con lineal (False si encontró un contraejemplo),
    falla (el primer contraejemplo, como texto), mensaje y, si no encontró contraejemplo, matriz. Es una PRUEBA, no
    una demostración: puede descubrir que f no es lineal, pero no garantizar que lo sea."""
    if isinstance(n, bool) or not isinstance(n, (int, np.integer)) or not 1 <= int(n) <= 6:
        raise ValueError("n, el número de componentes del vector de entrada, debe ser un entero de 1 a 6")
    n = int(n)
    tam = [None]

    def T(v):
        try:
            y = f(v.copy())
            y = sp.Matrix(y) if isinstance(y, (list, tuple, sp.MatrixBase, np.ndarray)) else sp.Matrix([y])
        except Exception as err:
            raise ValueError(f"la función no se pudo evaluar en x = {_tupla(v)} ({type(err).__name__}: {err})") from None
        y = y.reshape(len(y), 1)
        if len(y) == 0 or not all(sp.sympify(e).is_number for e in y):
            raise ValueError(f"T{_tupla(v)} = {_tupla(y)} no es un vector de números")
        if tam[0] is None:
            tam[0] = y.rows
        elif y.rows != tam[0]:
            raise ValueError(f"la función da vectores de tamaños distintos ({tam[0]} y {y.rows} componentes)")
        return y

    def falla(texto):
        return {"lineal": False, "falla": texto, "mensaje": f"no es lineal: {texto}", "matriz": None}

    T0 = T(sp.zeros(n, 1))
    if not _iguales(T0, sp.zeros(T0.rows, 1)):
        return falla(f"T(0) = {_tupla(T0)}, no es 0")
    base = [sp.eye(n)[:, j] for j in range(n)]
    prueba = list(base)
    R = sp.Rational
    for v in (sp.Matrix(list(range(1, n + 1))), sp.Matrix([3, -1, 4, -2, 5, -3][:n]),
              sp.Matrix([R(1, 2), R(-1, 3), R(2, 5), R(-1, 7), R(3, 11), R(-2, 13)][:n])):
        if v not in prueba:
            prueba.append(v)
    for i, u in enumerate(prueba):
        for w in prueba[i + 1:]:
            izq, der = T(u + w), T(u) + T(w)
            if not _iguales(izq, der):
                return falla(f"T(u + v) ≠ T(u) + T(v) con u = {_tupla(u)} y v = {_tupla(w)}: "
                             f"T(u + v) = {_tupla(izq)} y T(u) + T(v) = {_tupla(der)}")
    for c in (2, -3, R(1, 2), R(-1, 3)):
        for u in prueba:
            izq, der = T(c * u), c * T(u)
            if not _iguales(izq, der):
                cc = _paren_txt(c)
                return falla(f"T({cc}·u) ≠ {cc}·T(u) con u = {_tupla(u)}: T({cc}·u) = {_tupla(izq)} y "
                             f"{cc}·T(u) = {_tupla(der)}")
    A = sp.Matrix.hstack(*[T(e) for e in base])
    for u in prueba:
        if not _iguales(T(u), A * u):
            return falla(f"T(u) ≠ A·u con u = {_tupla(u)}, A = [T(e1) ... T(en)]")
    return {"lineal": True, "falla": None, "mensaje": _SIN_CONTRAEJEMPLO, "matriz": A}


def _paren_txt(c):
    """Un escalar para escribir delante de '·u': 2 -> '2', 1/2 -> '(1/2)', -3 -> '(-3)'."""
    return str(c) if (c.is_integer and c >= 0) else f"({c})"


# --- dibujo de una transformación del plano -----------------------------------------------------
_CASA = np.array([(0, 0), (2, 0), (2, 1.5), (1, 2.3), (0, 1.5)], dtype=float)
_CUADRADO = np.array([(0, 0), (1, 0), (1, 1), (0, 1)], dtype=float)


def _dibuja_en(ax, A, figura="casa", titulo=None):
    """Dibuja en ax la figura original (punteada) y su imagen bajo A (rellena gris), e1 y T(e1) (rotulados en la
    leyenda, para que el rótulo no se encime en la figura); si rango < 2, segmentos finos discontinuos en la
    dirección del núcleo: los puntos de cada uno caen en el mismo lugar."""
    Af = _flotante(A)
    F = _CASA if figura == "casa" else _CUADRADO
    G = F @ Af.T
    ni = nucleo_imagen(A)
    cerrada = lambda P: np.vstack([P, P[:1]])
    ax.fill(*G.T, facecolor="0.8", edgecolor="black", lw=1.8, hatch="..", alpha=0.9, label=f"imagen de la {figura}")
    ax.plot(*cerrada(G).T, "-", color="black", lw=1.8)
    ax.plot(*cerrada(F).T, ":", color="black", lw=2.0, label=f"{figura} original")
    e1, Te1 = np.array([1.0, 0.0]), Af @ np.array([1.0, 0.0])
    L = 1.2 * max(1.0, float(np.abs(np.vstack([F, G, Te1])).max())) + 0.4
    if ni["rango"] == 1:
        d = _flotante(ni["base_nucleo"][0]).ravel()
        d = d / np.linalg.norm(d)
        for k, p in enumerate(F):
            ax.plot(*np.array([p - 3 * L * d, p + 3 * L * d]).T, "--", color="0.45", lw=0.8,
                    label="dirección del núcleo: cada línea cae en un solo punto" if k == 0 else None)
    _flecha(ax, (0, 0), e1, _COLORES[0], "--", 2.2)
    _flecha(ax, (0, 0), Te1, _COLORES[1], "-", 2.6)
    ax.plot([], [], "--", color=_COLORES[0], lw=2.2, label="e1 (flecha discontinua)")
    ax.plot([], [], "-", color=_COLORES[1], lw=2.6, label="T(e1) (flecha continua)")
    ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_aspect("equal")
    ax.axhline(0, color="black", lw=0.6); ax.axvline(0, color="black", lw=0.6)
    ax.grid(alpha=0.4); ax.set_xlabel("x"); ax.set_ylabel("y")
    if titulo is None:
        titulo = {2: "rango 2: la figura cambia de tamaño, forma u orientación, pero no se aplasta",
                  1: "rango 1: la figura se aplasta sobre una recta (el núcleo es una recta)",
                  0: "rango 0: todo va a dar al origen"}[ni["rango"]]
    ax.set_title(textwrap.fill(titulo, 55), fontsize=10)
    ax.legend(loc="lower left", fontsize=8)


def dibuja_transformacion(A, figura="casa"):
    """Dibuja en el plano una casa (o un cuadrado) y su imagen bajo la transformación de matriz A (de 2×2)."""
    A = _matriz_t(A)
    if A.shape != (2, 2):
        raise ValueError(f"el dibujo es para transformaciones del plano: A debe ser de 2×2 (esta es {dims(A)})")
    if figura not in ("casa", "cuadrado"):
        raise ValueError("la figura debe ser «casa» o «cuadrado»")
    fig, ax = plt.subplots(figsize=(6.2, 6.2))
    _dibuja_en(ax, A, figura)
    plt.show()

**Recorrido paso a paso.** Primero, $R(30^\circ)$ aplicada a $(2,0)$: el resultado exacto es $(\sqrt3,1)$. Después, la proyección sobre la recta de $\mathbf u=(3,4)$: $P=\tfrac1{25}\begin{bmatrix}9&12\\12&16\end{bmatrix}$; $(5,0)$ va a $(9/5,12/5)$, $(4,-3)$ (perpendicular a $\mathbf u$) va al origen y $P^2=P$. La reflexión respecto a esa recta manda $(5,0)$ a $(-7/5,24/5)$. Al final, dos rotaciones seguidas: $R(20^\circ)R(25^\circ)=R(45^\circ)$, y los dibujos de la casa bajo la rotación y bajo la proyección (que la aplasta sobre una recta).

In [ ]:
_R30 = rotacion(30, "grados")
muestra("R(30 grados)", _R30, 4, tex=r"R(30^\circ)")
print("R(30 grados)·(2, 0) =", _tupla(_R30 * vector("2, 0"), 4))
print()
_P = proyeccion("3, 4")
muestra("P", _P, 4)
print("P = (1/25)·" + _en_linea(25 * _P))
print("P·(5, 0) =", _tupla(_P * vector("5, 0"), 4), "  P·(4, -3) =", _tupla(_P * vector("4, -3")),
      "  ¿P² = P?", "sí" if _P * _P == _P else "no")
_H = reflexion("3, 4")
muestra("H = 2P - I", _H, 4)
print("H·(5, 0) =", _tupla(_H * vector("5, 0"), 4), "  det H =", _H.det(),
      "  ¿H² = I?", "sí" if _H * _H == sp.eye(2) else "no")
print()
_C = sp.simplify(rotacion(20, "grados") * rotacion(25, "grados"))
muestra("R(20 grados)·R(25 grados)", _C, 4, tex=r"R(20^\circ)\,R(25^\circ)")
print("¿R(20 grados)·R(25 grados) = R(45 grados)?", "sí" if _C == rotacion(45, "grados") else "no")
dibuja_transformacion(_R30)
dibuja_transformacion(_P);

**¿Es lineal?** `prueba_linealidad(f, n)` busca un contraejemplo con vectores enteros y no enteros. La función del transmisor de 4 a 20 mA, $I=4+0.16\,T$, falla desde $T(\mathbf 0)$; $T(x,y)=(x^2,y)$ falla la aditividad; $T(x)=\lfloor x\rfloor$ (parte entera) pasa las pruebas con enteros, pero no con $\tfrac12$. $T(x,y)=(2x-y,\ x+3y)$ no da ningún contraejemplo y la función devuelve su matriz estándar; ojo: es una **prueba**, no una demostración (la demostración es verificar las dos propiedades con letras). La misma matriz sale de `matriz_desde_imagenes`, con $T(\mathbf e_1)$ y $T(\mathbf e_2)$ escritos uno por renglón (como lista de vectores: cada renglón se vuelve una columna).

In [ ]:
print(prueba_linealidad(lambda v: sp.Matrix([4 + sp.Rational(4, 25) * v[0]]), 1)["mensaje"])
print(prueba_linealidad(lambda v: sp.Matrix([v[0] ** 2, v[1]]), 2)["mensaje"])
print(prueba_linealidad(lambda v: sp.Matrix([sp.floor(v[0])]), 1)["mensaje"])
_lin = prueba_linealidad(lambda v: sp.Matrix([2 * v[0] - v[1], v[0] + 3 * v[1]]), 2)
print(_lin["mensaje"][0].upper() + _lin["mensaje"][1:] + ":", _en_linea(_lin["matriz"]))
_A_img = matriz_desde_imagenes("2, 1; -1, 3")          # T(e1) = (2, 1) y T(e2) = (-1, 3), uno por renglón
print("Con las imágenes T(e1) = (2, 1) y T(e2) = (-1, 3) como columnas:", _en_linea(_A_img),
      "(la misma)" if _A_img == _lin["matriz"] else "(¡distinta!)")

**Calculadora.** Elige el tipo de transformación y sus datos: para la rotación, el ángulo **y su unidad** (en radianes puedes escribir `pi/6`, `2*pi/3` o `π/4`); para el escalamiento, $k_1$ y $k_2$; para la proyección y la reflexión, la dirección $\mathbf u$ de la recta; para la matriz propia, la matriz **por renglones**. Los controles que no usa el tipo elegido se ignoran. Prueba la rotación con 30 en radianes y compara con 30 grados.

In [ ]:
_TIPOS = ["rotación", "escalamiento", "proyección sobre una recta", "reflexión respecto a una recta", "matriz propia"]


def _tex_tupla(v):
    return r"\left(" + ",\\ ".join(_latex(e) for e in v) + r"\right)"


def _transformacion(tipo, angulo, unidad, k1, k2, u_txt, M_txt):
    """(A, nombre, tex, avisos) de la transformación elegida en la calculadora."""
    avisos = []
    if tipo == "rotación":
        u_ang = _unidad(unidad)
        A = rotacion(angulo, u_ang)
        nombre, tex = f"R({_texto_angulo(angulo, u_ang)})", rf"R\left({_tex_angulo(angulo, u_ang)}\right)"
        if u_ang == "radianes" and abs(angulo_exacto(angulo)) > 2 * sp.pi:
            avisos.append(f"escribiste {_texto_angulo(angulo, u_ang)}, más de una vuelta completa (2π ≈ 6.283 radianes). "
                          "¿Seguro que está en radianes? Si lo pensaste en grados, cambia la unidad a «grados».")
    elif tipo == "escalamiento":
        A = escalamiento(k1, k2)
        nombre = f"S = diag({_txt(A[0, 0])}, {_txt(A[1, 1])})"
        tex = rf"S=\operatorname{{diag}}\left({_latex(A[0, 0])},\ {_latex(A[1, 1])}\right)"
    elif tipo in ("proyección sobre una recta", "reflexión respecto a una recta"):
        u = _direccion(u_txt)
        if u.rows != 2:
            raise ValueError(f"en el plano, la dirección u debe tener 2 componentes (escribiste {u.rows})")
        if tipo.startswith("proyección"):
            A, letra, frase = proyeccion(u), "P", "proyección sobre la recta de"
        else:
            A, letra, frase = reflexion(u), "H", "reflexión respecto a la recta de"
        nombre = f"{letra} ({frase} u = {_tupla(u)})"
        tex = rf"{letra}\ \text{{({frase} }}\mathbf u={_tex_tupla(u)}\text{{)}}"
    elif tipo == "matriz propia":
        A, nombre, tex = matriz_limitada(M_txt, "la matriz propia"), "A (matriz propia)", r"A\ \text{(matriz propia)}"
    else:
        raise ValueError(f"tipo desconocido: «{tipo}»; elige uno de: {', '.join(_TIPOS)}")
    return A, nombre, tex, avisos


def _vec_texto(v, n_cifras):
    """Un vector exacto con sus decimales; si lo exacto es muy largo (raíces anidadas), solo los decimales."""
    return _tupla(v, n_cifras) if _exacta_legible(v) else "≈ " + _aprox(v, n_cifras)


def _vec_base(v, n_cifras):
    """Un vector de una base, para mostrarlo: si es racional, escalado a enteros sin factor común (como en las
    secciones 1 y 2); si no, exacto o en decimales."""
    if all(e.is_Rational for e in v) and any(e != 0 for e in v):
        return _tupla(_primitivo(v))
    return _vec_texto(_simplifica(v), n_cifras)


def _informe_transformacion(A, nombre, x, n_cifras, tex=None):
    m, n = A.shape
    corto = nombre.split(" (")[0].split(" = ")[0]
    muestra(nombre, A, n_cifras, tex=tex)
    print(f"T va de {_R(n)} a {_R(m)}: T(x) = {corto}·x.")
    if m == n:
        d = sp.simplify(A.det())
        if not _exacta_legible(sp.Matrix([d])):
            print(f"det {corto} ≈ {cifras(d, n_cifras)}")
        else:
            print(f"det {corto} = {_txt(d)}" + ("" if d.is_Rational else f" ≈ {cifras(d, n_cifras)}"))
    else:
        print("det: solo existe para matrices cuadradas.")
    ni = nucleo_imagen(A)
    print(f"rango = {ni['rango']}")
    nucleo = ni["base_nucleo"]
    if not nucleo:
        print("Núcleo: {0} (dim 0): solo el 0 va al 0.")
    else:
        print(f"Núcleo: gen{{{', '.join(_vec_base(v, n_cifras) for v in nucleo)}}} (dim {len(nucleo)}): " +
              ("ese vector y sus múltiplos van al 0." if len(nucleo) == 1 else
               "esos vectores y sus combinaciones van al 0."))
    if ni["dim_imagen"] == m:
        print(f"Imagen: todo {_R(m)} (dim {m}).")
    elif not ni["base_imagen"]:
        print("Imagen: {0} (dim 0).")
    else:
        print(f"Imagen: gen{{{', '.join(_vec_base(v, n_cifras) for v in ni['base_imagen'])}}} (dim {ni['dim_imagen']}).")
    print(f"Teorema de la dimensión: dim Nul + dim Im = {ni['dim_nucleo']} + {ni['dim_imagen']} = {n} = n")
    Tx = _simplifica(A * x)
    print(f"T(x) = {corto}·{_tupla(x)} " + ("= " if _exacta_legible(Tx) else "") + _vec_texto(Tx, n_cifras))
    return ni


def calculadora_transformacion(tipo, angulo, unidad, k1, k2, u_txt, M_txt, x_txt, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; el dibujo va después
        try:
            A, nombre, tex, avisos = _transformacion(tipo, angulo, unidad, k1, k2, u_txt, M_txt)
            x = vector(x_txt, "x")
            if x.rows != A.cols:
                raise ValueError(f"x tiene {_componentes(x.rows)} y la matriz tiene {A.cols} "
                                 f"{'columna' if A.cols == 1 else 'columnas'}: deben coincidir")
        except ValueError as err:
            print("Revisa la entrada:", err); return
        for aviso in avisos:
            print("Aviso:", aviso)
        _informe_transformacion(A, nombre, x, n_cifras, tex)
    if A.shape == (2, 2):
        dibuja_transformacion(A)
    else:
        print(f"El dibujo de la casa es para matrices de 2×2 (transformaciones del plano); esta es {dims(A)}.")


_CONTROLES_4A = dict(
    tipo=widgets.Dropdown(options=_TIPOS, value="rotación", description="tipo:"),
    angulo=widgets.Text(value="30", description="ángulo:", continuous_update=False),
    unidad=widgets.ToggleButtons(options=["grados", "radianes"], value="grados", description="unidad:"),
    k1=widgets.FloatText(value=1.5, description="k1:", continuous_update=False),
    k2=widgets.FloatText(value=0.5, description="k2:", continuous_update=False),
    u_txt=widgets.Text(value="3, 4", description="u =", continuous_update=False),
    M_txt=widgets.Textarea(value="1, 1; 0, 1", description="matriz:", continuous_update=False,
                           layout=widgets.Layout(width="300px", height="80px")),
    x_txt=widgets.Text(value="2, 0", description="x =", continuous_update=False),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_transformacion, **_CONTROLES_4A);

**Calculadora de composición.** Elige $T_1$ (actúa primero) y $T_2$ (actúa después). La matriz de «primero $T_1$, después $T_2$» es $A_2A_1$; la calculadora la compara con $A_1A_2$ y dibuja la casa con los dos órdenes.

In [ ]:
_SENCILLAS = {
    "rotación de 45°": lambda: rotacion(45, "grados"),
    "rotación de -30°": lambda: rotacion(-30, "grados"),
    "escalamiento diag(2, 1)": lambda: escalamiento(2, 1),
    "corte [[1, 1], [0, 1]]": lambda: matriz("1, 1; 0, 1"),
    "reflexión respecto al eje y": lambda: reflexion("0, 1"),
    "proyección sobre el eje y": lambda: proyeccion("0, 1"),
}


def calculadora_composicion(T1, T2, n_cifras):
    A1, A2 = _SENCILLAS[T1](), _SENCILLAS[T2]()
    P21, P12 = _simplifica(A2 * A1), _simplifica(A1 * A2)
    igual = all(_es_cero(e) for e in P21 - P12)
    with bloque():
        print(f"T1 = {T1} (actúa primero), T2 = {T2} (actúa después).")
        muestra("A1", A1, n_cifras)
        muestra("A2", A2, n_cifras)
        muestra("A2·A1 (primero T1, después T2)", P21, n_cifras,
                tex=r"A_2A_1\ \text{(primero }T_1\text{, después }T_2\text{)}")
        muestra("A1·A2 (primero T2, después T1)", P12, n_cifras,
                tex=r"A_1A_2\ \text{(primero }T_2\text{, después }T_1\text{)}")
        print("Coinciden: con estas dos, el orden no importa." if igual else
              "NO coinciden: el orden importa (el producto de matrices no es conmutativo).")
    fig, ejes = plt.subplots(1, 2, figsize=(11, 5.6))
    _dibuja_en(ejes[0], P21, titulo="primero T1, después T2: A2·A1")
    _dibuja_en(ejes[1], P12, titulo="primero T2, después T1: A1·A2")
    plt.tight_layout()
    plt.show()


_CONTROLES_4B = dict(
    T1=widgets.Dropdown(options=list(_SENCILLAS), value="rotación de 45°", description="T1 (primero):",
                        style={"description_width": "initial"}),
    T2=widgets.Dropdown(options=list(_SENCILLAS), value="reflexión respecto al eje y", description="T2 (después):",
                        style={"description_width": "initial"}),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_composicion, **_CONTROLES_4B);

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
_x20 = vector("2, 0")
_NO_LINEALES = [(lambda v: sp.Matrix([sp.floor(v[0])]), 1),                       # parte entera
                (lambda v: sp.Matrix([v[0] + sp.sin(2 * sp.pi * v[0])]), 1),       # x + sen(2πx)
                (lambda v: sp.Matrix([v[0] * sp.cos(sp.pi * v[1]) ** 2]), 2)]      # x·cos²(πy)
PRUEBAS_4 = [
    ("R(90 grados) = [[0, -1], [1, 0]]",
     lambda: rotacion(90, "grados") == sp.Matrix([[0, -1], [1, 0]])),
    ("R(30 grados)·(2, 0) = (√3, 1) exacto",
     lambda: rotacion(30, "grados") * _x20 == sp.Matrix([sp.sqrt(3), 1])),
    ("R(30 radianes) ≠ R(30 grados) y su entrada [0, 0] ≈ 0.1543 (cos 30 rad)",
     lambda: rotacion(30, "radianes") != rotacion(30, "grados")
             and cifras(rotacion(30, "radianes")[0, 0], 4) == "0.1543"),
    ("R(25 grados)·R(35 grados) = R(60 grados)",
     lambda: sp.simplify(rotacion(25, "grados") * rotacion(35, "grados") - rotacion(60, "grados")) == sp.zeros(2, 2)
             and np.allclose(_flotante(rotacion(25, "grados") * rotacion(35, "grados")),
                             _flotante(rotacion(60, "grados")), atol=1e-12, rtol=0)),
    ("proyección sobre u = (3, 4): P = (1/25)[[9, 12], [12, 16]]; P(5, 0) = (9/5, 12/5); P(4, -3) = 0; P² = P",
     lambda: (lambda P: P == sp.Matrix([[9, 12], [12, 16]]) / 25 and P * vector("5, 0") == vector("9/5, 12/5")
                        and P * vector("4, -3") == sp.zeros(2, 1) and P * P == P)(proyeccion("3,4"))),
    ("reflexión respecto a la recta de (3, 4): H(5, 0) = (-7/5, 24/5) y det H = -1",
     lambda: (lambda H: H * vector("5, 0") == vector("-7/5, 24/5") and H.det() == -1)(reflexion("3,4"))),
    ("núcleo e imagen de [[1, 0, 0], [0, 1, 0]]: rango 2, núcleo (0, 0, 1), dim imagen 2, 1 + 2 = 3",
     lambda: (lambda r: r["rango"] == 2 and len(r["base_nucleo"]) == 1
                        and sp.Matrix.hstack(r["base_nucleo"][0], vector("0, 0, 1")).rank() == 1
                        and r["dim_imagen"] == 2 and r["dim_nucleo"] + r["dim_imagen"] == r["n"] == 3)(
         nucleo_imagen("1,0,0; 0,1,0"))),
    ("T(v) = (v1 + v2, 2) no es lineal: falla en T(0)",
     lambda: (lambda r: not r["lineal"] and "T(0)" in r["falla"])(prueba_linealidad(lambda v: sp.Matrix([v[0] + v[1], 2]), 2))),
    ("T(v) = (v1², v2) no es lineal (falla la aditividad o la homogeneidad)",
     lambda: (lambda r: not r["lineal"] and ("T(u + v)" in r["falla"] or "·u)" in r["falla"]))(
         prueba_linealidad(lambda v: sp.Matrix([v[0] ** 2, v[1]]), 2))),
    ("T(v) = (2v1 - v2, v1 + 3v2): sin contraejemplo, con matriz [[2, -1], [1, 3]]",
     lambda: (lambda r: r["lineal"] and r["matriz"] == sp.Matrix([[2, -1], [1, 3]]))(
         prueba_linealidad(lambda v: sp.Matrix([2 * v[0] - v[1], v[0] + 3 * v[1]]), 2))),
    ("transmisor de 4 a 20 mA, I = 4 + 0.16 T: no es lineal",
     lambda: not prueba_linealidad(lambda v: sp.Matrix([4 + sp.Rational(4, 25) * v[0]]), 1)["lineal"]),
    ("R_z(90 grados)·(1, 0, 0) = (0, 1, 0) y det = 1",
     lambda: (lambda Rz: Rz * vector("1, 0, 0") == vector("0, 1, 0") and Rz.det() == 1)(rotacion3("z", 90, "grados"))),
    ("mano derecha: R_y(90 grados)·(1, 0, 0) = (0, 0, -1) y R_x(90 grados)·(0, 1, 0) = (0, 0, 1)",
     lambda: rotacion3("y", 90, "grados") * vector("1, 0, 0") == vector("0, 0, -1")
             and rotacion3("x", 90, "grados") * vector("0, 1, 0") == vector("0, 0, 1")),
    ("escalamiento(2, 3)·(1, 1) = (2, 3) (k1 en x, k2 en y)",
     lambda: escalamiento(2, 3) * vector("1, 1") == vector("2, 3")),
    ("imagen de [[1, 0, 1], [2, 1, 3], [3, 1, 4]]: columnas ORIGINALES (1, 2, 3) y (0, 1, 1)",
     lambda: nucleo_imagen("1, 0, 1; 2, 1, 3; 3, 1, 4")["base_imagen"] == [vector("1, 2, 3"), vector("0, 1, 1")]),
    ("matriz_desde_imagenes: T(e1) = (2, 1), T(e2) = (-1, 3) → [[2, -1], [1, 3]] (las imágenes son columnas)",
     lambda: matriz_desde_imagenes("2, 1; -1, 3") == sp.Matrix([[2, -1], [1, 3]])),
    ("sin unidad no hay rotación: rotacion(30) y rotacion3(\"z\", 90) → ValueError «indica la unidad»",
     lambda: _mensaje(lambda: rotacion(30)).startswith("indica la unidad")
             and _mensaje(lambda: rotacion3("z", 90)).startswith("indica la unidad")),
    ("ángulos con pi en radianes: «pi/6» = 30 grados, «π/4» = 45 grados, «2*pi/3» = 120 grados; «pi/6» en grados → ValueError",
     lambda: rotacion("pi/6", "radianes") == rotacion(30, "grados") and rotacion("π/4", "radianes") == rotacion(45, "grados")
             and rotacion("2*pi/3", "radianes") == rotacion(120, "grados") and _rechaza(lambda: rotacion("pi/6", "grados"))),
    ("prueba_linealidad con no enteros: parte entera, x + sen(2πx) y x·cos²(πy) no son lineales",
     lambda: all(not prueba_linealidad(f, n)["lineal"] for f, n in _NO_LINEALES)
             and "no se encontró contraejemplo" in prueba_linealidad(lambda v: sp.Matrix([3 * v[0]]), 1)["mensaje"]),
    ("prueba de humo: la calculadora corre con sus valores iniciales, con cada uno de los cinco tipos",
     lambda: all(_humo(calculadora_transformacion, _CONTROLES_4A, tipo=t) for t in _TIPOS)),
    ("prueba de humo: la calculadora de composición corre con sus valores iniciales",
     lambda: _humo(calculadora_composicion, _CONTROLES_4B)),
]
corre_pruebas(PRUEBAS_4, 4)

**Contrasta (sección 4).** Calcula a mano la imagen de $\mathbf x=(2,0)$ al rotarlo $60^\circ$ en sentido antihorario: $R(60^\circ)\,\mathbf x=(2\cos 60^\circ,\ 2\operatorname{sen} 60^\circ)$. Escribe `mi_imagen = (a, b)` con dos decimales.

In [ ]:
mi_imagen = None     # dos números con dos decimales, por ejemplo (0.25, -1.50)

if mi_imagen is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mia = vector(mi_imagen, "tu imagen")
    except ValueError as err:
        print(f"Revisa tu imagen: {str(err).rstrip('.')}.")
    else:
        norma = math.hypot(*[float(e) for e in mia]) if mia.rows == 2 else None
        if mia.rows != 2:          # respuesta imposible: la referencia todavía no se muestra
            print(f"La imagen de un vector del plano tiene dos componentes (escribiste {mia.rows}): revisa tu "
                  "respuesta y vuelve a ejecutar la celda.")
        elif abs(norma - 2) > 0.05:     # una rotación no cambia la longitud: tampoco se muestra la referencia
            print(f"Una rotación no cambia la longitud: la imagen de (2, 0) debe medir 2, y la tuya mide "
                  f"{cifras(norma, 3)}. Revisa tu cálculo y vuelve a ejecutar la celda.")
        else:
            ref = rotacion(60, "grados") * vector("2, 0")
            dos = lambda v: "(" + ", ".join(decimales_fijos(e, 2) for e in v) + ")"
            print(f"La calculadora da: R(60 grados)·(2, 0) = {_tupla(ref)} ≈ {dos(ref)}.")
            con_radianes = rotacion(60, "radianes") * vector("2, 0")
            if all(abs(float(a - b)) <= 0.005 for a, b in zip(mia, ref)):
                print("Coinciden (con tolerancia 0.005).")
            elif all(abs(float(a - b)) <= 0.02 for a, b in zip(mia, con_radianes)):
                print(f"NO coinciden: tu resultado se parece a {dos(con_radianes)}, que sale si la calculadora "
                      "está en radianes (cos 60 rad ≈ -0.952). Probablemente usaste radianes: pon tu calculadora en grados.")
            else:
                print("NO coinciden: cos 60° = 1/2 y sen 60° = √3/2 ≈ 0.866; multiplica cada uno por 2.")
            print("\nEl procedimiento completo:")
            _informe_transformacion(rotacion(60, "grados"), "R(60 grados)", vector("2, 0"), 4, tex=r"R(60^\circ)")

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
_ESPERADAS = {1: 10, 2: 9, 3: 15, 4: 21}          # pruebas por sección (el libro cita estos números)
fallas, total = [], 0
for k, pruebas in enumerate([PRUEBAS_1, PRUEBAS_2, PRUEBAS_3, PRUEBAS_4], 1):
    assert len(pruebas) == _ESPERADAS[k], (f"la sección {k} tiene {len(pruebas)} pruebas y deben ser "
                                           f"{_ESPERADAS[k]}: falta o sobra alguna")
    resultados = evalua(pruebas)                      # cada caso se ejecuta una sola vez
    print(f"Sección {k}: {sum(ok for _, ok in resultados)}/{len(pruebas)}")
    fallas += [f"sección {k}: {nombre}" for nombre, ok in resultados if not ok]
    total += len(pruebas)
for falla in fallas:
    print("FALLA", falla)
assert not fallas, f"Fallan {len(fallas)} de {total} pruebas (están arriba)"
print(f"Todas las verificaciones pasaron: {total}/{total}")